### Update log

last update: 8/31/2025 6:22pm
* added atrocious cpp implementation
* thanks chatgpt @ nebius playground

last update: 5/12/2025 11:11pm
* added progress bars
* removing redundancies now goes from the end by default
* made dynamic programming in recursive path order only engage for words longer than 20 letters, also fixed a bug in recursive path order that prevented it from actually using alphabet_order
* added code for B(2,3) and B(2,4)
* B(2,4) should take 2-4 hours to run

last update: 5/7/2025 9:38pm
* added naive dynamic programming implementation of recursive_path_order
* it's a bit faster than recursion
* dynamic programming implementation now used by default in recursive_path_order

last update: 5/6/2025 1:04pm
* added positive first alphabet order and all 6 respective options for word order:
  - (shortlex, recurseive)x(default, chenadec, positive first)
* some of the 6 option resolve every example, including braid group

## C++ Knuth-Bendix implementation

In [ ]:
%%writefile ordering.cpp

#ifndef ORDERING_HPP
#define ORDERING_HPP

#include <vector>
#include <functional>
#include <cstdlib>      // std::abs
#include <algorithm>    // std::min
#include <cassert>

/* -------------------------------------------------------------
   Helper type aliases
   ------------------------------------------------------------- */
using Word           = std::vector<int>;
using AlphabetOrder  = std::function<int(int, int)>;
using Rule = std::pair<Word, Word>;
using Rules = std::vector<Rule>;

/* -------------------------------------------------------------
   1  Basic alphabet orders
   ------------------------------------------------------------- */

/*  x_1 < x_1⁻¹ < x_2 < x_2⁻¹ < ...  */
inline int alphabet_order(int a, int b)
{
    if (std::abs(a) > std::abs(b))      return  1;
    if (std::abs(a) < std::abs(b))      return -1;
    if (a < b)                          return -1;
    if (a > b)                          return  1;
    return 0;
}

/*  x_1 < x_2 < ... < x_n < x_1⁻¹ < x_2⁻¹ < ...  */
inline int alphabet_order_positive_first(int a, int b)
{
    if (a > 0 && b < 0)                 return -1;
    if (a < 0 && b > 0)                 return  1;

    if (a > 0 && b > 0) {               // both positive
        if (a > b) return  1;
        if (a < b) return -1;
        return 0;
    }

    if (a < 0 && b < 0) {               // both negative
        if (a < b) return  1;           // note the reversed direction
        if (a > b) return -1;
        return 0;
    }

    return 0;   // should never be reached
}

/*  Chenadec order
    x_2p > x_2p⁻¹ > x_{2p-2} > ... > x_2⁻¹ > x_1 > x_1⁻¹ > ... > x_{2p-1}⁻¹   */
inline int chenadec_order(int a, int b)
{
    const bool a_even = (a % 2 == 0);
    const bool b_even = (b % 2 == 0);

    if (a_even && !b_even)  return  1;
    if (!a_even && b_even)  return -1;

    if (a_even && b_even) {               // both even
        if (std::abs(a) > std::abs(b))   return  1;
        if (std::abs(a) < std::abs(b))   return -1;
        if (a > b)                       return  1;
        if (a < b)                       return -1;
        return 0;
    }

    // both odd
    if (std::abs(a) < std::abs(b))       return  1;
    if (std::abs(a) > std::abs(b))       return -1;
    if (a > b)                           return  1;
    if (a < b)                           return -1;
    return 0;
}

/* -------------------------------------------------------------
   2  Word orderings that use an alphabet order
   ------------------------------------------------------------- */

/* Short-lex : first compare length, then compare letter-by-letter
   using the supplied alphabet order                                 */
inline int shortlex(const Word& w1,
                    const Word& w2,
                    const AlphabetOrder& alph = alphabet_order)
{
    if (w1.size() > w2.size()) return  1;
    if (w1.size() < w2.size()) return -1;

    for (std::size_t i = 0; i < w1.size(); ++i) {
        int c = alph(w1[i], w2[i]);
        if (c != 0) return c;   // either 1 or -1
    }
    return 0;
}

/* Helper wrappers that bind a concrete alphabet order               */
inline int shortlex_default(const Word& a, const Word& b)
{
    return shortlex(a, b);
}
inline int shortlex_with_chenadec(const Word& a, const Word& b)
{
    return shortlex(a, b, chenadec_order);
}
inline int shortlex_with_positive_first(const Word& a, const Word& b)
{
    return shortlex(a, b, alphabet_order_positive_first);
}

/* Pure lexicographic (no length check first) - be careful!          */
inline int lex(const Word& w1,
               const Word& w2,
               const AlphabetOrder& alph = alphabet_order)
{
    const std::size_t m = std::min(w1.size(), w2.size());
    for (std::size_t i = 0; i < m; ++i) {
        int c = alph(w1[i], w2[i]);
        if (c != 0) return c;
    }
    if (w1.size() > w2.size()) return  1;
    if (w1.size() < w2.size()) return -1;
    return 0;
}

/* -------------------------------------------------------------
   3  Recursive Path Order (RPO)
   ------------------------------------------------------------- */

/* Forward declaration (used by the DP version) */
inline int recursive_path_order_dyn_prog(const Word& w1,
                                          const Word& w2,
                                          const AlphabetOrder& alph);

/* The classic recursive definition (with optional DP shortcut).    */
inline int recursive_path_order(const Word& w1,
                                const Word& w2,
                                const AlphabetOrder& alph = alphabet_order,
                                bool use_dynprog = true)
{
    /* DP optimisation - only used when at least one word is “big”.
       The original Python code used >20; we keep the same threshold   */
    if (use_dynprog && (w1.size() > 20 || w2.size() > 20))
        return recursive_path_order_dyn_prog(w1, w2, alph);

    /* Base cases ---------------------------------------------------- */
    if (w1.empty() && w2.empty()) return 0;
    if (w1.empty())               return -1;   // ε < anything non-empty
    if (w2.empty())               return  1;

    if (w1.size() == 1 && w2.size() == 1)
        return alph(w1[0], w2[0]);

    /* Main recursive cases ------------------------------------------ */
    if (w1[0] == w2[0]) {
        Word w1_tail(w1.begin() + 1, w1.end());
        Word w2_tail(w2.begin() + 1, w2.end());
        return recursive_path_order(w1_tail, w2_tail, alph, use_dynprog);
    }

    // case 2:  w1[0] > w2[0]  and  w1 > w2[1..]
    if (alph(w1[0], w2[0]) == 1) {
        Word w2_tail(w2.begin() + 1, w2.end());
        if (recursive_path_order(w1, w2_tail, alph, use_dynprog) == 1)
            return 1;
    }

    // case 3:  w1[1..] >= w2
    Word w1_tail(w1.begin() + 1, w1.end());
    int ord = recursive_path_order(w1_tail, w2, alph, use_dynprog);
    if (ord == 1 || ord == 0) return 1;
    return -1;
}

/* -------------------------------------------------------------
   4  Iterative (dynamic-programming) version of RPO
   ------------------------------------------------------------- */
inline int recursive_path_order_dyn_prog(const Word& w1,
                                          const Word& w2,
                                          const AlphabetOrder& alph)
{
    /* Trivial one-letter words - keep the same behaviour as the
       recursive version                                            */
    if (w1.size() == 1 && w2.size() == 1)
        alph(w1[0], w2[0]);
    if (w1.empty() && w2.empty()) return 0;
    if (w1.empty())               return -1;
    if (w2.empty())               return  1;

    const std::size_t n = w1.size();
    const std::size_t m = w2.size();

    /* DP table: dp[i][j] stores the result of RPO( w1[i..], w2[j..] )
       i ranges 0..n, j ranges 0..m.  The extra row/column correspond to
       the empty suffix.                                             */
    std::vector<std::vector<int>> dp(n + 1, std::vector<int>(m + 1, 0));

    /* Initialise the “border” according to the Python code */
    for (std::size_t j = 0; j < m; ++j) dp[n][j] = -1;   // ε > non-empty suffix of w2
    for (std::size_t i = 0; i < n; ++i) dp[i][m] =  1;   // non-empty suffix of w1 > ε
    dp[n][m] = 0;                                       // ε = ε

    /* Fill the table by anti-diagonals (i + j = const) */

    for (int diag = static_cast<int>(n + m - 2); diag >= 0; --diag) {
        /* i runs from max(0, diag-m+1) up to min(diag, n-1) */
        const int i_start = std::max(0, diag - static_cast<int>(m) + 1);
        const int i_end   = std::min(diag, static_cast<int>(n) - 1);

        for (int i = i_start; i <= i_end; ++i) {
            const int j = diag - i;            // because i + j = diag
            assert(j >= 0 && static_cast<std::size_t>(j) < m);

            if (w1[i] == w2[j]) {
                dp[i][j] = dp[i + 1][j + 1];
            }
            else if (alph(w1[i], w2[j]) == 1 && dp[i][j + 1] == 1) {
                dp[i][j] = 1;
            }
            else if (dp[i + 1][j] >= 0) {
                dp[i][j] = 1;
            }
            else {
                dp[i][j] = -1;
            }
        }
    }

    return dp[0][0];
}

/* -------------------------------------------------------------
   5  Convenience wrappers for the specialised orders
   ------------------------------------------------------------- */
/*
inline int recursive_path_order_with_chenadec(const Word& a,
                                               const Word& b,
                                               bool use_dynprog = true)
{
    return recursive_path_order(a, b, chenadec_order, use_dynprog);
}
inline int recursive_path_order_with_positive_first(const Word& a,
                                                     const Word& b,
                                                     bool use_dynprog = true)
{
    return recursive_path_order(a, b,
                                alphabet_order_positive_first,
                                use_dynprog);
}
*/

inline int recursive_path_order_with_chenadec(const Word& a,
                                               const Word& b)
                                               //bool use_dynprog = true)
{
    return recursive_path_order(a, b, chenadec_order, true);
}
inline int recursive_path_order_with_positive_first(const Word& a,
                                                     const Word& b)
                                                     //bool use_dynprog = true)
{
    return recursive_path_order(a, b,
                                alphabet_order_positive_first,
                                true);
}

/* -------------------------------------------------------------
   6  Placeholder for “wreath order” (not implemented in the
        original Python file)
   ------------------------------------------------------------- */
inline int wreath_order(const Word&, const Word&,
                        const AlphabetOrder& = alphabet_order)
{
    return 0;   // stub
}




#endif // ORDERING_HPP

Overwriting ordering.cpp


In [ ]:
%%writefile vector.cpp

#include <iostream>
#include <list>
#include <vector>
#include <utility>

// Helper function to print a vector of integers
void print_vector(const std::vector<int>& vec) {
    std::cout << "{";
    for (size_t i = 0; i < vec.size(); ++i) {
        std::cout << vec[i] << (i == vec.size() - 1 ? "" : ", ");
    }
    std::cout << "}";
}

// Helper function to print a pair of vectors of integers
void print_pair_of_vectors(const std::pair<std::vector<int>, std::vector<int>>& p) {
    std::cout << "{";
    print_vector(p.first);
    std::cout << ", ";
    print_vector(p.second);
    std::cout << "}";
}


Writing vector.cpp


In [ ]:
%%writefile reduce.cpp
#include <iostream>
#include <vector>
#include <utility>

using Word = std::vector<int>;
using Rule = std::pair<Word, Word>;
using Rules = std::vector<Rule>;

Word apply_rule(Word word,
                const std::pair<Word, Word>& rule,
                bool& applied,
                std::size_t place = 0)
{
    const Word& lhs = rule.first;   // pattern we are looking for
    const Word& rhs = rule.second;  // pattern we will substitute

    // Guard against out‑of‑range start positions.
    if (place > word.size())
        return word;

    // Walk through *word* looking for a match.
    // The loop stops when there aren’t enough remaining elements to hold *lhs*.
    for (std::size_t i = place;
         i + lhs.size() <= word.size(); ++i)
    {
        // Compare the sub‑range [i, i+lhs.size()) with *lhs*.
        // std::equal works on iterators and stops as soon as a mismatch is found.
        if (std::equal(lhs.begin(), lhs.end(),
                       word.begin() + i))
        {
            applied = true;
            // -----  Replacement  -----
            // 1. Erase the old segment (lhs)
            word.erase(word.begin() + i,
                       word.begin() + i + lhs.size());

            // 2. Insert the new segment (rhs) at the same position
            word.insert(word.begin() + i,
                        rhs.begin(), rhs.end());

            // The rule has been applied once – stop (or continue if you want
            // multiple replacements, just remove the `break`).
            break;
        }
    }

    return word;
}

// ------------------------------------------------------------------
//  reduce
// ------------------------------------------------------------------
//  * Repeatedly scans the list of rewrite *rules* and applies the first
//    applicable rule at the first possible position (the behaviour of
//    the original Python version).
//  * *skip_rule* can be used to temporarily ignore a rule – the default
//    value -1 means “do not skip any rule”.  It is useful for the
//    redundancy‑check of the Knuth‑Bendix algorithm.
//  * If *print_progress* is true a short message is printed each
//    iteration, mirroring the Python `print("new round...")`.
//  * The function returns the (fully) reduced word.
// ------------------------------------------------------------------
Word reduce(Word word,
           const Rules& rules,
           int skip_rule = -1,
           bool print_progress = false)
{
    bool applied = false;
    while (true)
    {
        applied = false;
        if (print_progress)
            std::cout << "new round...\n";

        // Keep a copy so we can detect whether anything changed this round.
        // const Word starting_word = word;

        // Try each rule in order (except the one we are asked to skip).
        for (std::size_t i = 0; i < rules.size(); ++i)
        {
            if (static_cast<int>(i) == skip_rule)
                continue;                       // <-- skip this rule

            // apply_rule returns a new word; if a rule matches it will be
            // different from the input; otherwise it is unchanged.
            word = apply_rule(word, rules[i],applied);
        }

        // If no rule altered the word during the whole pass we are done.
        if (!applied)
            break;
    }
    return word;
}

// reorder_rules not quite working
void reorder_rules(Rules& rules, std::function<int(Word, Word)> &order)
{
  //'' for each rule u->v, rewrite it as u->v or v->u according to order '''
  for (std::size_t i = 0; i < rules.size(); ++i)   // i is an unsigned index
  {
    if(order(rules[i].first, rules[i].second) == -1)
      swap(rules[i].first,rules[i].second);
  }
}


Writing reduce.cpp


In [ ]:
%%writefile main.cpp
#include "ordering.cpp"
#include "reduce.cpp"
#include <iostream>
#include <utility>
#include <algorithm>

int main()
{
    // Two words (as vectors of integers)
    Word a = { 1, -2, 3, -4 };
    Word b = { 1, -2, 3 };
    Word w = {1, 2, 3, 4, 5, 2, 3, 6};
    Word lhs = {2, 3};
    Word rhs = {9, 9, 9};

    auto rule = std::make_pair(lhs, rhs);
    Rules rules = { rule };

    lhs = {9, 9};
    rhs = {10};
    rule = std::make_pair(lhs, rhs);
    rules.push_back(rule);

    reorder_rules(rules,shortlex);

    for (int n : rules[0].first) std::cout << n << ' ';
    for (int n : rules[0].second) std::cout << n << ' ';
    for (int n : rules[1].first) std::cout << n << ' ';
    for (int n : rules[1].second) std::cout << n << ' ';

    bool applied = false;

    // Word result = apply_rule(w, rule, applied);
    // std::cout << "applied: " << applied << '\n';
    Word result = reduce(w, rules);   // replaces the *first* occurrence

    for (int n : result) std::cout << n << ' ';
    // Expected output: 1 9 9 9 4 5 2 3 6
    std::cout << '\n';

    std::cout << "shortlex(a,b)                 = " << shortlex(a,b)                 << '\n';
    std::cout << "shortlex_with_chenadec(a,b)   = " << shortlex_with_chenadec(a,b)   << '\n';
    std::cout << "lex(a,b)                      = " << lex(a,b)                      << '\n';
    std::cout << "recursive_path_order(a,b)     = " << recursive_path_order(a,b)     << '\n';
    std::cout << "recursive_path_order_chenadec = " <<
              recursive_path_order_with_chenadec(a,b) << '\n';
}

Overwriting main.cpp


In [ ]:
%%writefile crit_pairs.cpp
/***************************************************************************************************
 *  Re‑implementation of the Python functions
 *        make_crit_pairs   – builds the list of critical pairs
 *        check_confluence  – tests a rewriting system for confluence
 *
 *  All data structures are vectors (dynamic arrays) and pairs of vectors.
 **************************************************************************************************/

#include <algorithm>      // std::equal, std::move
#include <iostream>       // std::cout, std::cerr
#include <iterator>       // std::back_inserter
#include <utility>        // std::pair
#include <vector>         // std::vector
#include <cstddef>        // std::size_t

// -----------------------------------------------------------------------------
//  Type aliases (exactly the same as in the previous answers)
using Word  = std::vector<int>;                     // a word = list of ints
using Rule  = std::pair<Word, Word>;                // lhs → rhs
using Rules = std::vector<Rule>;                    // whole rewriting system
using CritPair  = std::pair<Word, Word>;            // one critical pair
using CritPairs = std::vector<CritPair>;            // list of critical pairs
// -----------------------------------------------------------------------------

// -----------------------------------------------------------------------------
//  Helper utilities (concatenation, printing, …)
// -----------------------------------------------------------------------------
static inline Word concat(const Word& a, const Word& b)
{
    Word r; r.reserve(a.size() + b.size());
    r.insert(r.end(), a.begin(), a.end());
    r.insert(r.end(), b.begin(), b.end());
    return r;
}

static inline void print_word(const Word& w)
{
    std::cout << '[';
    for (std::size_t i = 0; i < w.size(); ++i) {
        std::cout << w[i];
        if (i + 1 < w.size()) std::cout << ' ';
    }
    std::cout << ']';
}

// -----------------------------------------------------------------------------
//  Accessors for the two components of a rule – keep the same name as the Python
//  helpers so the translation stays close to the original source.
// -----------------------------------------------------------------------------
static inline const Word& rule_left (const Rule& r) { return r.first;  }
static inline const Word& rule_right(const Rule& r) { return r.second; }

// -----------------------------------------------------------------------------
//  Forward declaration of `reduce`.  The definition is the one that was fixed in the
//  previous answer – it returns a (possibly) reduced word and tells whether a rule
//  was applied via the `applied` flag.
// -----------------------------------------------------------------------------
//Word reduce(Word word,
//           const Rules& rules,
//           int skip_rule = -1,
//           bool print_progress = false);   // implementation follows after make_crit_pairs

// ===========================================================================
//  make_crit_pairs
// ===========================================================================
/**
 * @brief Build the list of critical pairs for a given set of rules.
 *
 * @param rules                the rewriting system (vector of pairs)
 * @param crit_pairs           container that will receive the pairs.
 * @param reduce_immediately   if true, each candidate pair is reduced immediately;
 *                             only pairs whose two reductions differ are stored.
 * @param start_at_rule        only pairs that involve a rule with index ≥ this value
 *                             are examined (used for incremental Knuth‑Bendix).
 * @param max_crit_length      0 -> no length limit; otherwise discard pairs whose
 *                             concatenated word would be longer than this.
 * @param reset_pairs          if true, clear `crit_pairs` before filling it.
 * @param print_progress       verbose printing of each candidate pair.
 * @param print_progress_pct   fake “percentage” mode; we simply iterate with a
 *                             different loop variable – the original used tqdm.
 *
 * @return reference to the filled `crit_pairs` (for convenience)
 */
CritPairs& make_crit_pairs(const Rules& rules,
                          CritPairs& crit_pairs,
                          bool reduce_immediately = true,
                          std::size_t start_at_rule = 0,
                          std::size_t max_crit_length = 0,
                          bool reset_pairs = true,
                          bool print_progress = false,
                          bool print_progress_pct = false)
{
    if (reset_pairs) crit_pairs.clear();

    /* ----------------------------------------------------------------------
     *  1️  Overlap of the left‑hand sides
     * ---------------------------------------------------------------------- */
    if (print_progress_pct) std::cout << "Building crit pairs – overlap:\n";

    auto i_range = (print_progress_pct ? std::vector<std::size_t>{} : std::vector<std::size_t>{});
    // we do not really need a fancy progress bar – a plain loop is fine.
    for (std::size_t i = 0; i < rules.size(); ++i) {
        for (std::size_t j = 0; j < rules.size(); ++j) {
            if (i < start_at_rule && j < start_at_rule) continue;

            const Word& lhs1 = rule_left (rules[i]);
            const Word& lhs2 = rule_left (rules[j]);

            std::size_t max_k = std::min(lhs1.size(), lhs2.size());
            for (std::size_t k = 1; k < max_k; ++k) {                // overlap size = k
                // lhs1[-k:] must equal lhs2[:k]
                if (!std::equal(lhs1.end() - k, lhs1.end(), lhs2.begin())) continue;

                // length check (optional)
                if (max_crit_length != 0 &&
                    lhs1.size() + lhs2.size() - k > max_crit_length) continue;

                // Build the two words that will be compared
                Word w1 = concat(rule_right(rules[i]), Word(lhs2.begin() + k, lhs2.end()));
                Word w2 = concat(Word(lhs1.begin(), lhs1.end() - k), rule_right(rules[j]));

                if (reduce_immediately) {
                    Word r1 = reduce(w1, rules);
                    Word r2 = reduce(w2, rules);
                    if (print_progress) {
                        std::cout << "Overlap: ";
                        print_word(concat(lhs1, Word(lhs2.begin() + k, lhs2.end())));
                        std::cout << "  →  ";
                        print_word(r1); std::cout << " , ";
                        print_word(r2); std::cout << '\n';
                    }
                    if (r1 != r2) crit_pairs.emplace_back(std::move(r1), std::move(r2));
                } else {
                    if (print_progress) {
                        std::cout << "Overlap (raw): ";
                        print_word(concat(lhs1, Word(lhs2.begin() + k, lhs2.end())));
                        std::cout << "  vs  ";
                        print_word(w1); std::cout << " , ";
                        print_word(w2); std::cout << '\n';
                    }
                    crit_pairs.emplace_back(std::move(w1), std::move(w2));
                }
            }
        }
    }

    /* ----------------------------------------------------------------------
     *  2️  Inclusion of one lhs into another
     * ---------------------------------------------------------------------- */
    if (print_progress_pct) std::cout << "Building crit pairs – inclusion:\n";

    for (std::size_t i = 0; i < rules.size(); ++i) {
        const Word& lhs1 = rule_left(rules[i]);

        for (std::size_t j = start_at_rule; j < rules.size(); ++j) {
            if (i == j) continue;
            const Word& lhs2 = rule_left(rules[j]);

            // --------------------------------------------------------------
            //  (a) lhs2 longer (or equal) than lhs1 -> search lhs1 inside lhs2
            // --------------------------------------------------------------
            if ((max_crit_length == 0 || lhs2.size() <= max_crit_length) &&
                lhs2.size() >= lhs1.size())
            {
                for (std::size_t k = 0; k + lhs1.size() <= lhs2.size(); ++k) {
                    if (!std::equal(lhs1.begin(), lhs1.end(), lhs2.begin() + k)) continue;

                    Word w1 = concat(Word(lhs2.begin(), lhs2.begin() + k),
                                    concat(rule_right(rules[i]),
                                           Word(lhs2.begin() + k + lhs1.size(), lhs2.end())));
                    Word w2 = rule_right(rules[j]);

                    if (reduce_immediately) {
                        Word r1 = reduce(w1, rules);
                        Word r2 = reduce(w2, rules);
                        if (print_progress) {
                            std::cout << "Inclusion (raw): ";
                            print_word(lhs2); std::cout << " >= ";
                            print_word(lhs1); std::cout << "  -->  ";
                            print_word(r1); std::cout << " , ";
                            print_word(r2); std::cout << '\n';
                        }
                        if (r1 != r2) crit_pairs.emplace_back(std::move(r1), std::move(r2));
                    } else {
                        if (print_progress) {
                            std::cout << "Inclusion (raw): ";
                            print_word(lhs2); std::cout << " >= ";
                            print_word(lhs1); std::cout << "  -->  ";
                            print_word(w1); std::cout << " , ";
                            print_word(w2); std::cout << '\n';
                        }
                        crit_pairs.emplace_back(std::move(w1), std::move(w2));
                    }
                }
            }
            // --------------------------------------------------------------
            //  (b) lhs1 longer than lhs2 – we only need to consider the
            //      asymmetric case i < start_at_rule && j >= start_at_rule
            // --------------------------------------------------------------
            else if (i < start_at_rule && j >= start_at_rule &&
                     (max_crit_length == 0 || lhs1.size() <= max_crit_length) &&
                     lhs1.size() > lhs2.size())
            {
                for (std::size_t k = 0; k + lhs2.size() <= lhs1.size(); ++k) {
                    if (!std::equal(lhs2.begin(), lhs2.end(), lhs1.begin() + k)) continue;

                    Word w1 = concat(Word(lhs1.begin(), lhs1.begin() + k),
                                    concat(rule_right(rules[j]),
                                           Word(lhs1.begin() + k + lhs2.size(), lhs1.end())));
                    Word w2 = rule_right(rules[i]);

                    if (reduce_immediately) {
                        Word r1 = reduce(w1, rules);
                        Word r2 = reduce(w2, rules);
                        if (print_progress) {
                            std::cout << "Inclusion (swap): ";
                            print_word(lhs1); std::cout << " <= ";
                            print_word(lhs2); std::cout << "  -->  ";
                            print_word(r1); std::cout << " , ";
                            print_word(r2); std::cout << '\n';
                        }
                        if (r1 != r2) crit_pairs.emplace_back(std::move(r1), std::move(r2));
                    } else {
                        if (print_progress) {
                            std::cout << "Inclusion (swap): ";
                            print_word(lhs1); std::cout << " <= ";
                            print_word(lhs2); std::cout << "  -->  ";
                            print_word(w1); std::cout << " , ";
                            print_word(w2); std::cout << '\n';
                        }
                        crit_pairs.emplace_back(std::move(w1), std::move(w2));
                    }
                }
            }
        }
    }

    return crit_pairs;
}


// -----------------------------------------------------------------------------
//  check_confluence
// -----------------------------------------------------------------------------
bool check_confluence(const Rules&          rules,
                      CritPairs*           crit_pairs = nullptr,   // ← optional
                      std::size_t          max_crit_length = 0,
                      bool                 erase_pair_list = true,
                      bool                 print_progress = false)
{
    /* ---------------------------------------------------------------
     *  1️  Choose the container we will operate on.
     *
     *  If the caller supplied a pointer we use it directly.
     *  Otherwise we create a local `CritPairs` object that lives for the
     *  duration of this call.  The variable `cp` always points to a valid
     *  container.
     * --------------------------------------------------------------- */
    CritPairs local_storage;                 // only constructed when needed
    CritPairs* cp = crit_pairs ? crit_pairs : &local_storage;

    /* ---------------------------------------------------------------
     *  2️  If the container is empty we have to build the list on‑the‑fly.
     * --------------------------------------------------------------- */
    if (cp->empty()) {
        make_crit_pairs(rules, *cp,
                        /*reduce_immediately=*/true,
                        /*start_at_rule=*/0,
                        max_crit_length,
                        /*reset_pairs=*/true,
                        print_progress,
                        /*print_progress_pct=*/false);
    }

    /* ---------------------------------------------------------------
     *  3️  Main loop – test each critical pair.
     * --------------------------------------------------------------- */
    for (std::size_t i = 0; i < cp->size(); ++i) {
        const auto& cpair = (*cp)[i];

        if (print_progress) {
            std::cout << "testing pair " << i << ": ";
            print_word(cpair.first);  std::cout << "  ,  ";
            print_word(cpair.second); std::cout << '\n';
        }

        Word w1 = reduce(cpair.first,  rules, -1, false);
        Word w2 = reduce(cpair.second, rules, -1, false);

        if (w1 != w2) {
            if (print_progress) {
                std::cout << "Confluence failure for pair ";
                print_word(cpair.first);  std::cout << " , ";
                print_word(cpair.second); std::cout << "\n   → ";
                print_word(w1);           std::cout << " , ";
                print_word(w2);           std::cout << '\n';
            }
            if (erase_pair_list) cp->clear();
            return false;
        }
    }

    /* ---------------------------------------------------------------
     *  4️  Clean‑up – mirror the original semantics.
     * --------------------------------------------------------------- */
    if (erase_pair_list) cp->clear();
    return true;
}

/*
// ---------------------------------------------------------------------
//  a) Let the function create a temporary list (the original behaviour)
// ---------------------------------------------------------------------
bool ok1 = check_confluence(R);                     // all defaults
bool ok2 = check_confluence(R, nullptr, 0, true, true);

// ---------------------------------------------------------------------
//  b) Provide your own container – maybe you want to reuse it later
// ---------------------------------------------------------------------
CritPairs precomputed;
make_crit_pairs(R, precomputed, true, 0, 0, true, false, false); // optional
bool ok3 = check_confluence(R, &precomputed, 0, false, true);
*/

/*
bool check_confluence(const Rules& rules,
                      CritPairs& crit_pairs,
                      std::size_t max_crit_length = 0,
                      bool erase_pair_list = true,
                      bool print_progress = false)
{
    // If the list is empty we have to build it on‑the‑fly (default behaviour
    // mirrors the Python version).
    if (crit_pairs.empty()) {
        make_crit_pairs(rules, crit_pairs,
                        true,
                        0,
                        max_crit_length,
                        true,
                        print_progress,
                        false);
    }

    for (std::size_t i = 0; i < crit_pairs.size(); ++i) {
        const auto& cp = crit_pairs[i];

        if (print_progress) {
            std::cout << "testing pair " << i << ": ";
            print_word(cp.first); std::cout << "  ,  ";
            print_word(cp.second); std::cout << '\n';
        }

        Word w1 = reduce(cp.first,  rules, -1, false);
        Word w2 = reduce(cp.second, rules, -1, false);

        if (w1 != w2) {
            if (print_progress) {
                std::cout << "Confluence failure for pair ";
                print_word(cp.first); std::cout << " , ";
                print_word(cp.second); std::cout << "\n   → ";
                print_word(w1); std::cout << " , ";
                print_word(w2); std::cout << '\n';
            }
            if (erase_pair_list) crit_pairs.clear();
            return false;
        }
    }

    if (erase_pair_list) crit_pairs.clear();
    return true;
}
*/





Writing crit_pairs.cpp


In [ ]:
%%writefile resolve.cpp
/******************************************************************************************
 *  Knuth‑Bendix – critical‑pair resolution
 *
 *  This file contains the C++ equivalents of the two Python functions
 *      resolve_crit_pair
 *      resolve_all_crit_pairs
 *
 *  --------------------------------------------------------------
 *  Conventions used in the translation
 *  --------------------------------------------------------------
 *  - A *word*                      : std::vector<int>
 *  - A *rule* (left → right)       : std::pair<std::vector<int>, std::vector<int>>
 *  - The whole rule set            : std::vector<rule>
 *  - A *critical pair* (w1 , w2)   : std::pair<std::vector<int>, std::vector<int>>
 *  - The whole critical‑pair list  : std::vector<critical_pair>
 *
 *  All functions that are called from here (reduce, make_crit_pairs,
 *  shortlex, …) are assumed to be implemented elsewhere.
 *
 *  --------------------------------------------------------------
 *  Author : (your name)
 *  Date   : 2025‑08‑21
 ******************************************************************************************/

#include <iostream>
#include <vector>
#include <utility>      // std::pair
#include <functional>   // std::function
#include <algorithm>    // std::remove_if

/* -------------------------------------------------------------------------- *
 *  Forward declarations of the external helpers
 * -------------------------------------------------------------------------- */
using CPairs = std::vector<CritPair>;
using Word  = std::vector<int>;                     // a word = list of ints
using Rule  = std::pair<Word, Word>;                // lhs → rhs
using Rules = std::vector<Rule>;                    // whole rewriting system
using CritPair  = std::pair<Word, Word>;            // one critical pair
using CritPairs = std::vector<CritPair>;            // list of critical pairs

/* -------------------------------------------------------------------------- *
 *  resolve_crit_pair
 * -------------------------------------------------------------------------- */
/**
 * @brief Resolve one critical pair.
 *
 * @param rules          The (mutable) list of current rules.
 * @param crit_pairs     The mutable list of critical pairs.
 * @param idx            Index of the critical pair that shall be resolved.
 * @param order          Ordering function (e.g. shortlex). Must return 1,
 *                       0 or -1.
 * @param print_progress If true, progress messages are printed.
 * @return int           1 if a new rule was added, 0 otherwise.
 */
int resolve_crit_pair(std::vector<Rule>&               rules,
                      CPairs&                          crit_pairs,
                      std::size_t                      idx,
                      const std::function<int(const Word&, const Word&)>& order,
                      bool                             print_progress = false)
{
    if (print_progress) {
        std::cout << "Resolving " << "["
                  << crit_pairs[idx].first.size() << ", "
                  << crit_pairs[idx].second.size() << "]\n";
    }

    // Reduce both sides of the critical pair.
    Word w1 = reduce(crit_pairs[idx].first,  rules, print_progress);
    Word w2 = reduce(crit_pairs[idx].second, rules, print_progress);

    // If the reductions differ we have to add a new rule.
    if (w1 != w2) {
        // Order decides which direction the rule will have.
        if (order(w1, w2) == 1) {
            // w1 > w2  ->  w1 -> w2
            rules.emplace_back(std::move(w1), std::move(w2));
        } else {
            // w2 >= w1 -> w2 -> w1
            rules.emplace_back(std::move(w2), std::move(w1));
        }

        // Remove the processed critical pair.
        crit_pairs.erase(crit_pairs.begin() + static_cast<std::ptrdiff_t>(idx));

        if (print_progress) {
            const Rule& added = rules.back();
            std::cout << "Added rule: [";
            for (int x : added.first) std::cout << x << ' ';
            std::cout << "-> ";
            for (int x : added.second) std::cout << x << ' ';
            std::cout << "]\n";
        }
        return 1;
    }

    // No rule necessary – just erase the pair.
    crit_pairs.erase(crit_pairs.begin() + static_cast<std::ptrdiff_t>(idx));

    if (print_progress) {
        std::cout << "No rule added\n";
    }
    return 0;
}

/* -------------------------------------------------------------------------- *
 *  resolve_all_crit_pairs
 * -------------------------------------------------------------------------- */
/**
 * @brief Resolve *all* critical pairs (optionally generated on the fly).
 *
 * @param rules                The mutable rule set.
 * @param crit_pairs           Optional pre‑computed critical‑pair list.
 *                             If empty, it will be generated with
 *                             make_crit_pairs().
 * @param order                Ordering function (default = shortlex).
 * @param reduce_crit_immediately  Passed to make_crit_pairs().
 * @param go_from_the_end     If true resolve from the back of the list,
 *                             otherwise from the front.
 * @param print_progress      If true, progress information is printed.
 * @param print_crit_pairs_progress  Passed to make_crit_pairs().
 * @return bool                true if at least one new rule was added.
 */
bool resolve_all_crit_pairs(std::vector<Rule>&               rules,
                            CPairs                           crit_pairs = CPairs(),
                            const std::function<int(const Word&, const Word&)>& order = shortlex_default,
                            bool                             reduce_crit_immediately = true,
                            bool                             go_from_the_end = true,
                            bool                             print_progress = false,
                            bool                             print_crit_pairs_progress = false)
{
    // If no critical pairs were supplied, generate them.
    if (crit_pairs.empty()) {
        crit_pairs = make_crit_pairs(rules,
                                     crit_pairs,
                                     /* bool reduce_immediately = */ reduce_crit_immediately,
                                     /* std::size_t start_at_rule = */ 0,
                                     /* std::size_t max_crit_length = */ 0,
                                     /* bool reset_pairs = */ true,
                                     /* bool print_progress = */ print_progress,
                                     /* bool print_progress_pct = */ print_crit_pairs_progress);
    }

    int rule_counter = 0;

    if (print_progress) {
        std::cout << "Resolving critical pairs (" << crit_pairs.size() << " total)...\n";
    }

    // Main loop – keep going until the list is empty.
    while (!crit_pairs.empty()) {
        std::size_t idx = go_from_the_end ? crit_pairs.size() - 1 : 0;
        rule_counter += resolve_crit_pair(rules,
                                          crit_pairs,
                                          idx,
                                          order,
                                          print_progress);
    }

    if (print_progress) {
        std::cout << "Finished resolving critical pairs.\n";
    }

    if (rule_counter > 0) {
        if (print_progress) {
            std::cout << "Added " << rule_counter << " new rule"
                      << (rule_counter == 1 ? "" : "s") << "\n";
        }
        return true;
    }
    return false;
}

/* -------------------------------------------------------------------------- *
 *  Example of how the functions could be used (not part of the translation)
 * -------------------------------------------------------------------------- */
// int main() {
//     std::vector<Rule> rules;
//     // ... fill `rules` somehow ...
//     bool changed = resolve_all_crit_pairs(rules, {}, shortlex, true, true, true, false);
//     std::cout << "Any new rules? " << std::boolalpha << changed << '\n';
//     return 0;
// }


Writing resolve.cpp


In [ ]:
%%writefile shortcut.cpp
#include <vector>
#include <utility>   // std::pair
#include <cstddef>   // std::size_t
#include <algorithm> // std::move

// --------------------------------------------------------------------------
//  Type aliases (as requested)
// --------------------------------------------------------------------------
using Word  = std::vector<int>;                     // a word = list of ints
using Rule  = std::pair<Word, Word>;                // lhs → rhs
using Rules = std::vector<Rule>;                    // whole rewriting system
using CritPair  = std::pair<Word, Word>;            // one critical pair
using CritPairs = std::vector<CritPair>;            // list of critical pairs

// --------------------------------------------------------------------------
//  1. shortcut – try to shorten the RHS of a single rule
// --------------------------------------------------------------------------
/**
 * @brief Reduces the right‑hand side of rule *rule_number* using the whole
 *        system *rules*.
 *
 * @param rules        the rewriting system (modified in‑place)
 * @param rule_number  index of the rule to try to shortcut
 *
 * @return true  if the RHS was changed (a shortcut was made)
 * @return false otherwise
 */
bool shortcut(Rules& rules, std::size_t rule_number)
{
    // obtain the two sides of the selected rule
    const Word& lhs = rule_left (rules[rule_number]);
    const Word& rhs = rule_right(rules[rule_number]);

    // reduce the RHS with respect to the whole system
    Word reduced_rhs = reduce(rhs, rules);

    // if the reduction changed anything, replace the RHS
    if (reduced_rhs != rhs) {
        // In the original Python code the rule was stored as a single list
        // (len(lhs), lhs..., rhs...).  With the `Rule = pair<Word,Word>`
        // representation we simply overwrite the second element.
        rules[rule_number].second = std::move(reduced_rhs);
        return true;
    }
    return false;
}

// --------------------------------------------------------------------------
//  2. make_all_shortcuts – try to shortcut every rule in the system
// --------------------------------------------------------------------------
/**
 * @brief Reduces the RHS of **all** rules once.
 *
 * @param rules  the rewriting system (modified in‑place)
 *
 * @return true  if at least one rule was shortened
 * @return false otherwise
 */
bool make_all_shortcuts(Rules& rules)
{
    bool changes_made = false;
    for (std::size_t i = 0; i < rules.size(); ++i) {
        // the logical‑or ensures that once *true* it stays true
        changes_made = shortcut(rules, i) || changes_made;
    }
    return changes_made;
}

Writing shortcut.cpp


In [ ]:
%%writefile redundancy.cpp
#include <vector>
#include <utility>      // std::pair
#include <cstddef>      // std::size_t
#include <algorithm>    // std::move
#include <optional>

// -----------------------------------------------------------------------------
//  1. is_rule_redundant
// -----------------------------------------------------------------------------
/// @brief  Checks whether rule *rule_number* is redundant.
///
/// A rule is redundant when its left‑hand side and right‑hand side reduce to
/// the same word when **all other** rules are available for rewriting.
///
/// @param rules        the whole system (read‑only)
/// @param rule_number  index of the rule to test
/// @return true  iff the rule is redundant
/// @return false otherwise
bool is_rule_redundant ( const Rules& rules, std::size_t rule_number )
{
    const Word& lhs = rule_left ( rules[rule_number] );
    const Word& rhs = rule_right( rules[rule_number] );

    // Reduce while *skipping* the rule we are testing
    Word lhs_red = reduce( lhs, rules, rule_number );
    Word rhs_red = reduce( rhs, rules, rule_number );

    return lhs_red == rhs_red;
}

// -----------------------------------------------------------------------------
//  2. eliminate_redundancy
// -----------------------------------------------------------------------------
/// @brief  Removes all redundant rules from *rules*.
///
/// The function can walk the vector from the beginning or from the end
/// (controlled by *go_from_the_end*).  While removing elements we have to keep
/// the index arithmetic that the original Python code relied on:
///   – when a rule is erased we do **not** advance the loop counter,
///   – the variable *new_first_rule* is adjusted whenever a rule that lies
///     before the current “first new rule” is deleted.
///
/// @param rules            the rewriting system (modified in‑place)
/// @param start_at_rule    the index of the first rule that will be used later
///                         when critical pairs are created (default = 0)
/// @param go_from_the_end  if true the vector is scanned from the back,
///                         otherwise from the front (default = true)
/// @return a pair *(redundancies_were_present, new_first_rule)*
///         – *redundancies_were_present* is true iff at least one rule was
///           eliminated,
///         – *new_first_rule* is the possibly‑shifted index to be returned to
///           the caller.
std::pair<bool, std::size_t>
eliminate_redundancy ( Rules&       rules,
                       std::size_t  start_at_rule     = 0,
                       bool         go_from_the_end  = true )
{
    bool        redundancies_were_present = false;
    std::size_t i                         = 0;
    std::size_t new_first_rule            = start_at_rule;

    if ( go_from_the_end )
    {
        // -------------------------------------------------------------
        // walk from the *end* (the Python code uses “len(rules)-1-i”)
        // -------------------------------------------------------------
        while ( i < rules.size() )
        {
            std::size_t idx = rules.size() - 1 - i;          // current rule

            if ( is_rule_redundant( rules, idx ) )
            {
                redundancies_were_present = true;

                if ( idx < new_first_rule )
                    --new_first_rule;                       // shift the start index

                // erase the redundant rule
                rules.erase( rules.begin() + static_cast<std::ptrdiff_t>( idx ) );
                // do NOT ++i : the vector became shorter, the next element
                // that slides into position *idx* must be examined in the
                // next iteration.
            }
            else
                ++i;                                         // move to the next element
        }
    }
    else
    {
        // -------------------------------------------------------------
        // walk from the *front*
        // -------------------------------------------------------------
        while ( i < rules.size() )
        {
            if ( is_rule_redundant( rules, i ) )
            {
                redundancies_were_present = true;

                // erase the rule at position i
                rules.erase( rules.begin() + static_cast<std::ptrdiff_t>( i ) );

                if ( i < new_first_rule )
                    --new_first_rule;                       // shift the start index
                // i stays the same – a new rule has slid into slot i
            }
            else
                ++i;                                         // advance
        }
    }

    return { redundancies_were_present, new_first_rule };
}


Writing redundancy.cpp


In [ ]:
%%writefile knuth-bendix.cpp
#include <iostream>
#include <vector>
#include <utility>
#include <functional>
#include <cassert>

// -----------------------------------------------------------------------------
//  knuth_bendix – the main completion loop
// -----------------------------------------------------------------------------
/**
 * @brief Runs the Knuth‑Bendix completion algorithm.
 *
 * @param rules                     the rewriting system (modified in‑place)
 * @param order                     total order on words (defaults to shortlex)
 * @param reduce_crit_immediately   whether a critical pair is reduced as soon as it is created
 * @param max_rounds                limit on the number of rounds (‑1 = no limit)
 * @param recheck_old_rules        if true, critical pairs are recomputed for *all* rules each round
 * @param print_progress            enable textual progress output
 * @param print_crit_pairs_progress enable progress output while generating critical pairs
 *
 * @return number of rounds performed, or –1 if the algorithm stopped because
 *         `max_rounds` was hit.
 */
int knuth_bendix(
    Rules& rules,
    const std::function<int(const Word&, const Word&)>& order = shortlex_default,
    bool reduce_crit_immediately = true,
    int max_rounds = -1,
    bool recheck_old_rules = false,
    bool print_progress = false,
    bool print_crit_pairs_progress = false)
{
    bool changes_made = true;                // the outer‑loop guard
    int  rounds       = 0;                   // how many complete rounds have been executed
    std::size_t first_new_rule = 0;          // index of the first rule that is “new” in the current round
    CritPairs crit_pairs;                    // container for the critical pairs of the current round

    while (changes_made) {
        ++rounds;

        /*--------------------------------------------------------------
         *  Max‑rounds check (first entry – after the round counter has
         *  been incremented but before any work of the round begins)
         *--------------------------------------------------------------*/
        if (max_rounds >= 0 && rounds > max_rounds) {
            //if (print_progress) {
                std::cout << "Max rounds reached: " << max_rounds << '\n';
                std::cout << "Making shortcuts and removing redundancies, then stopping.\n";
            //}
            return -1;                       // early termination
        }

        if (print_progress) {
            std::cout << "Round " << rounds << '\n';
        }

        /*--------------------------------------------------------------
         *  Flags for this round
         *--------------------------------------------------------------*/
        bool shortcuts_made          = false;
        bool redundancies_eliminated = false;
        bool new_rules_added        = false;

        /*--------------------------------------------------------------
         *  1)  Make all shortcuts (reduce RHS of every rule)
         *--------------------------------------------------------------*/
        shortcuts_made = make_all_shortcuts(rules);
        if (print_progress && shortcuts_made) {
            std::cout << "Shortcuts made. " << rules.size() << " rules total.\n";
        }

        /*--------------------------------------------------------------
         *  2)  Remove redundant rules
         *--------------------------------------------------------------*/
        std::tie(redundancies_eliminated, first_new_rule) =
            eliminate_redundancy(rules, first_new_rule);
        if (print_progress && redundancies_eliminated) {
            std::cout << "Redundancies eliminated. " << rules.size()
                      << " rules total.\n";
        }

        /*--------------------------------------------------------------
         *  Second max‑rounds check – the Python version repeats it after
         *  shortcuts/redundancy removal.
         *--------------------------------------------------------------*/
        if (max_rounds >= 0 && rounds > max_rounds) {
            if (print_progress) {
                std::cout << "Stopping because max rounds reached: " << max_rounds << '\n';
            }
            return -1;
        }

        /*--------------------------------------------------------------
         *  3)  Build the list of critical pairs.
         *      The list must be empty at the start of each round.
         *--------------------------------------------------------------*/
        assert(crit_pairs.empty());

        if (recheck_old_rules) {
            // recompute critical pairs for *all* rules
            make_crit_pairs(rules,
                            crit_pairs,
                            reduce_crit_immediately,
                            0,                           // start_at_rule
                            print_crit_pairs_progress,
                            print_progress);
        } else {
            // only pairs that involve at least one rule that is newer than
            // `first_new_rule`
            make_crit_pairs(rules,
                            crit_pairs,
                            reduce_crit_immediately,
                            first_new_rule,
                            print_crit_pairs_progress,
                            print_progress);
        }

        if (print_progress) {
            std::cout << "Crit pairs: " << crit_pairs.size() << '\n';
        }

        /*--------------------------------------------------------------
         *  4)  From now on every rule is considered “old”.
         *--------------------------------------------------------------*/
        first_new_rule = rules.size();

        /*--------------------------------------------------------------
         *  5)  Resolve all critical pairs (may add new rules)
         *--------------------------------------------------------------*/
        new_rules_added = resolve_all_crit_pairs(rules,
                                                 crit_pairs,
                                                 order,
                                                 reduce_crit_immediately,
                                                 print_progress);
        if (print_progress && new_rules_added) {
            std::cout << "New rules added. " << rules.size() << " rules total.\n";
        }

        /*--------------------------------------------------------------
         *  6)  Did anything change this round?
         *--------------------------------------------------------------*/
        changes_made = shortcuts_made || redundancies_eliminated || new_rules_added;

        if (print_progress) {
            std::cout << rules.size()
                      << " rules total at the end of round " << rounds << '\n';
        }

        /*--------------------------------------------------------------
         *  Prepare for the next iteration
         *--------------------------------------------------------------*/
        crit_pairs.clear();   // start fresh next round
    }

    return rounds;           // normal termination – the system is confluent
}

Writing knuth-bendix.cpp


In [ ]:
%%writefile group.cpp
#include <vector>
#include <utility>
#include <algorithm>
#include <functional>
#include <cassert>


// ---------------------------------------------------------------------------
//  1. group_inverse --------------------------------------------------------
//  Returns the word obtained by negating every generator and then reversing
// ---------------------------------------------------------------------------
inline Word group_inverse (const Word& w)
{
    Word inv;
    inv.reserve(w.size());
    // negate while we copy
    for (auto it = w.rbegin(); it != w.rend(); ++it)
        inv.push_back(-(*it));
    return inv;                // e.g. [a,b] → [-b,-a]
}

// ---------------------------------------------------------------------------
//  2. symmetrize_group_rule -----------------------------------------------
//  For a rule *i* (u → v) we create the word   u · v⁻¹   and then add
//  all cyclic shifts of that word and of its inverse as new rules.
//  The order predicate must behave like the Python version:
//        order(a,b)  ∈ {‑1,0,1}  (‑1 ⇒ a < b,  1 ⇒ a > b)
// ---------------------------------------------------------------------------
void symmetrize_group_rule ( Rules&                       rules,
                             std::size_t                  i,
                             const std::function<int(const Word&, const Word&)>& order = nullptr )
{
    assert(i < rules.size());

    // -----------------------------------------------------------------------
    //  the word w = lhs(rule_i) · (rhs(rule_i))⁻¹
    // -----------------------------------------------------------------------
    Word w = rule_left(rules[i]);                     // copy lhs
    Word rhs_inv = group_inverse(rule_right(rules[i]));
    w.insert(w.end(), rhs_inv.begin(), rhs_inv.end());   // concatenate

    // -----------------------------------------------------------------------
    //  first pass : w itself
    // -----------------------------------------------------------------------
    for (std::size_t shift = 0; shift < w.size(); ++shift) {

        // cyclic shift
        Word shifted;
        shifted.reserve(w.size());
        shifted.insert(shifted.end(), w.begin() + shift, w.end());
        shifted.insert(shifted.end(), w.begin(), w.begin() + shift);

        // all possible splittings of the shifted word
        for (std::size_t split = 0; split <= shifted.size(); ++split) {

            Word left (shifted.begin(), shifted.begin() + split);   // w₁
            Word right_suffix(shifted.begin() + split, shifted.end()); // suffix

            // w₂ = (suffix)⁻¹
            Word right = group_inverse(right_suffix);

            // decide orientation using the supplied order predicate
            int cmp = order ? order(left, right) : 0;   // order may be nullptr → do nothing
            if (cmp == 1) {
                rules.emplace_back(left,  right);
            } else if (cmp == -1) {
                rules.emplace_back(right, left);
            } else {
                // cmp == 0 : the two sides are equal – nothing to add
            }
        }
    }

    // -----------------------------------------------------------------------
    //  second pass : the inverse of w
    // -----------------------------------------------------------------------
    w = group_inverse(w);                 // now w = (u·v⁻¹)⁻¹ = v·u⁻¹

    for (std::size_t shift = 0; shift < w.size(); ++shift) {
        Word shifted;
        shifted.reserve(w.size());
        shifted.insert(shifted.end(), w.begin() + shift, w.end());
        shifted.insert(shifted.end(), w.begin(), w.begin() + shift);

        for (std::size_t split = 0; split <= shifted.size(); ++split) {

            Word left (shifted.begin(), shifted.begin() + split);
            Word right = group_inverse(Word(shifted.begin() + split, shifted.end()));

            int cmp = order ? order(left, right) : 0;
            if (cmp == 1) {
                rules.emplace_back(left,  right);
            } else if (cmp == -1) {
                rules.emplace_back(right, left);
            }
        }
    }
}

// ---------------------------------------------------------------------------
//  3. add_free_group_rules -------------------------------------------------
//  Adds the cancelling pairs  x·x⁻¹ → ε   and   x⁻¹·x → ε   for every generator
//  that occurs (in either sign) in the current system.
//
//  In the original Python code the rule was stored as a flat list
//  [2, a, b] where the leading 2 is the length of the lhs.
//  With the pair representation we store the rule as
//        lhs = {a,b} , rhs = {}.
// ---------------------------------------------------------------------------
void add_free_group_rules ( Rules& rules )
{
    int max_abs = 1;                     // smallest possible generator is ±1

    // -----------------------------------------------------------------------
    //  scan all existing rules and find the largest absolute generator
    // -----------------------------------------------------------------------
    for (const Rule& r : rules) {
        for (int x : r.first)  max_abs = std::max(max_abs, std::abs(x));
        for (int x : r.second) max_abs = std::max(max_abs, std::abs(x));
    }

    // -----------------------------------------------------------------------
    //  prepend the cancelling rules (the Python version inserted at the front)
    // -----------------------------------------------------------------------
    for (int i = max_abs; i >= 1; --i) {
        //  x  x^{-1}  → ε
        rules.insert(rules.begin(), Rule(Word{ i, -i }, Word{}));
        //  x^{-1}  x  → ε
        rules.insert(rules.begin(), Rule(Word{ -i, i }, Word{}));
    }
}

Writing group.cpp


In [ ]:
%%writefile print.cpp
#include <iostream>
#include <vector>
#include <utility>   // std::pair
#include <type_traits>

// ------------------------------------------------------------
// Helper: detect if a type already has an overload of operator<<
// ------------------------------------------------------------
template <typename, typename = void>
struct is_ostream_printable : std::false_type {};

template <typename T>
struct is_ostream_printable<
    T,
    std::void_t<decltype(std::declval<std::ostream&>() << std::declval<T>())>
> : std::true_type {};

template <typename T>
constexpr bool is_ostream_printable_v = is_ostream_printable<T>::value;

// ------------------------------------------------------------
// 1️ operator<< for std::vector<T>
// ------------------------------------------------------------
template <typename T>
std::enable_if_t<is_ostream_printable_v<T>, std::ostream&>
operator<<(std::ostream& os, const std::vector<T>& vec)
{
    os << '[';
    for (auto it = vec.begin(); it != vec.end(); ++it) {
        if (it != vec.begin()) os << ", ";
        os << *it;                // relies on T being printable
    }
    os << ']';
    return os;
}

// ------------------------------------------------------------
// 2️ operator<< for std::pair<A,B>
// ------------------------------------------------------------
template <typename A, typename B>
std::enable_if_t<
    is_ostream_printable_v<A> && is_ostream_printable_v<B>,
    std::ostream&
>
operator<<(std::ostream& os, const std::pair<A,B>& p)
{
    os << "( " << p.first << " , " << p.second << " )";
    return os;
}

// ------------------------------------------------------------
// 3️ Convenience printer for the exact type asked for
// ------------------------------------------------------------
using TargetType = std::vector<std::pair<std::vector<int>, std::vector<int>>>;

inline void print(const TargetType& data, std::ostream& os = std::cout)
{
    os << data << '\n';
}

// ------------------------------------------------------------
// Demo driver (you can remove main() when integrating elsewhere)
// ------------------------------------------------------------
/* int main()
{
    // Build a sample value
    TargetType data = {
        { {1, 2, 3}, {4, 5} },
        { {6}, {} },
        { {}, {7, 8, 9, 10} }
    };

    // Use the generic printer
    print(data);

    // Direct usage of operator<< also works
    std::cout << "Direct: " << data << "\n";

    return 0;
}
*/

Writing print.cpp


In [ ]:
%%writefile main.cpp
#include "ordering.cpp"
#include "vector.cpp"
#include "reduce.cpp"
#include "crit_pairs.cpp"
#include "resolve.cpp"
#include "shortcut.cpp"
#include "redundancy.cpp"
#include "knuth-bendix.cpp"
#include "group.cpp"
#include "print.cpp"


int main()
{
  // Abelian group on a, b
  Rules rules = {{{1,2,-1,-2},{}}}; //Z^2
  // CritPairs crit_pairs={};

  //rules.insert(rules.begin(), Rule(Word{1,2,-1,-2}, Word{}));
  add_free_group_rules(rules);
  std::cout << "Starting rules: " << rules << "\n";
  int rounds = knuth_bendix(rules, shortlex_default, true, 10, true, false, false);
  if(rounds != -1)
    std::cout << "Finished in" << rounds << "rounds.\nTotal number of rules:" << rules.size() << "\n";
  else
    std::cout << "Terminated by reaching max_rounds.\nTotal number of rules:" << rules.size() << "\n";
  std::cout << rules << "\n";
  std::cout << "Confluence:" << check_confluence(rules) << "\n"; // Pass the non-const object
  std::cout << "---------\n";

/*
# Abelian group on a, b
rules = [[4,1,2,-1,-2]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, recheck_old_rules = False, print_progress=False, print_crit_pairs_progress = False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")
*/
}

Overwriting main.cpp


## C++ Burnside groups implementation

In [ ]:
%%writefile word_generator.cpp
/********************************************************************
 *  group_word_generator.hpp
 *
 *  C++17 implementation of the Python functions
 *      generate_group_words(...)
 *      generate_all_words(...)
 ********************************************************************/

#ifndef GROUP_WORD_GENERATOR_HPP
#define GROUP_WORD_GENERATOR_HPP

#include <algorithm>   // std::find, std::reverse, std::remove_if
#include <iterator>    // std::back_inserter
#include <vector>
#include <iostream>    // optional, for debugging
#include <utility>

namespace grp {

/* ------------------------------------------------------------------
 *  Helper: inverse of a word.
 *  The inverse is obtained by reversing the word and flipping the sign
 *  of every generator.
 * ------------------------------------------------------------------ */
inline std::vector<int> group_inverse(const std::vector<int>& w)
{
    std::vector<int> inv;
    inv.reserve(w.size());
    for (auto it = w.rbegin(); it != w.rend(); ++it)
        inv.push_back(-(*it));
    return inv;
}

/* ------------------------------------------------------------------
 *  Helper: test whether a word is already stored in the container.
 *  (Linear search – identical to Python's `in` operator.)
 * ------------------------------------------------------------------ */
inline bool contains(const std::vector<std::vector<int>>& container,
                     const std::vector<int>& word)
{
    return std::find(container.begin(), container.end(), word) != container.end();
}

/* ------------------------------------------------------------------
 *  generate_group_words
 * ------------------------------------------------------------------ */
inline void generate_group_words(
        std::vector<std::vector<int>>& words,
        int number_gens = 2,
        int max_length  = 4,
        bool exact_length = false,
        bool remove_cyc_reducible = true,
        bool remove_shifts = true,
        bool remove_inverses = true)
{
    /* ----- 1. initialise length‑1 words --------------------------- */
    words.clear();
    for (int i = 1; i <= number_gens; ++i) {
        words.push_back({ i });
        words.push_back({ -i });
    }

    /* ----- 2. grow words up to max_length -------------------------- */
    for (int length = 2; length <= max_length; ++length) {
        std::vector<std::vector<int>> new_words;
        new_words.reserve(words.size() * number_gens * 2);   // rough reserve

        for (const auto& w : words) {
            if (static_cast<int>(w.size()) != length - 1) continue;

            for (int gen = 1; gen <= number_gens; ++gen) {
                if (w.back() !=  gen) {                     // can append -gen
                    std::vector<int> tmp = w;
                    tmp.push_back(-gen);
                    new_words.push_back(std::move(tmp));
                }
                if (w.back() != -gen) {                     // can append  gen
                    std::vector<int> tmp = w;
                    tmp.push_back(gen);
                    new_words.push_back(std::move(tmp));
                }
            }
        }
        // concatenate
        words.insert(words.end(),
                     std::make_move_iterator(new_words.begin()),
                     std::make_move_iterator(new_words.end()));
    }

    /* ----- 3. post‑processing -------------------------------------- */
    if (exact_length) {
        /* ---- keep only words of length max_length ----------------- */
        words.erase(
            std::remove_if(words.begin(), words.end(),
                [max_length](const std::vector<int>& w){ return static_cast<int>(w.size()) != max_length; }),
            words.end());

        if (remove_inverses) {
            for (size_t i = 0; i < words.size(); ++i) {
                const auto inv = group_inverse(words[i]);
                if (contains(words, inv))
                    words.erase(std::remove(words.begin(), words.end(), inv), words.end());
            }
        }

        if (remove_shifts) {
            for (size_t i = 0; i < words.size(); ++i) {
                const auto& w = words[i];
                const int n = static_cast<int>(w.size());
                for (int sh = 1; sh < n; ++sh) {
                    std::vector<int> shifted(w.begin() + sh, w.end());
                    shifted.insert(shifted.end(), w.begin(), w.begin() + sh);
                    if (shifted != w && contains(words, shifted))
                        words.erase(std::remove(words.begin(), words.end(), shifted), words.end());
                }
            }
        }
        return;        // exact_length mode ends here
    }

    /* ---- a) remove cyclically reducible words ------------------- */
    if (remove_cyc_reducible) {
        for (auto it = words.begin(); it != words.end(); ) {
            if (it->size() > 2 && (*it)[0] == -static_cast<int>((*it).back())) {
                it = words.erase(it);               // erase returns next iterator
            } else {
                ++it;
            }
        }
    }

    /* ---- b) remove inverse duplicates --------------------------- */
    if (remove_inverses) {
        for (size_t i = 0; i < words.size(); ++i) {
            const auto inv = group_inverse(words[i]);
            if (contains(words, inv)) {
                words.erase(std::remove(words.begin(), words.end(), inv), words.end());
            }
        }
    }

    /* ---- c) remove shifted duplicates -------------------------- */
    if (remove_shifts) {
        for (size_t i = 0; i < words.size(); ++i) {
            const auto& w = words[i];
            const int n = static_cast<int>(w.size());
            for (int sh = 1; sh < n; ++sh) {
                std::vector<int> shifted(w.begin() + sh, w.end());
                shifted.insert(shifted.end(), w.begin(), w.begin() + sh);
                if (shifted != w && contains(words, shifted))
                    words.erase(std::remove(words.begin(), words.end(), shifted), words.end());
            }
        }
    }

    /* ---- d) remove shifts of inverses (only if both options set) */
    if (remove_shifts && remove_inverses) {
        for (size_t i = 0; i < words.size(); ++i) {
            const auto inv = group_inverse(words[i]);
            const int n = static_cast<int>(inv.size());
            for (int sh = 0; sh < n; ++sh) {
                std::vector<int> shifted(inv.begin() + sh, inv.end());
                shifted.insert(shifted.end(), inv.begin(), inv.begin() + sh);
                if (shifted != words[i] && contains(words, shifted))
                    words.erase(std::remove(words.begin(), words.end(), shifted), words.end());
            }
        }
    }
}

/* ------------------------------------------------------------------
 *  generate_all_words
 *  (same as Python version but without any of the reduction steps)
 * ------------------------------------------------------------------ */
inline void generate_all_words(
        std::vector<std::vector<int>>& words,
        int number_gens = 2,
        int max_length  = 4,
        bool exact_length = false)      // kept for signature compatibility
{
    words.clear();
    for (int i = 1; i <= number_gens; ++i) {
        words.push_back({ i });
        words.push_back({ -i });
    }

    for (int length = 2; length <= max_length; ++length) {
        std::vector<std::vector<int>> new_words;
        new_words.reserve(words.size() * number_gens * 2);

        for (const auto& w : words) {
            if (static_cast<int>(w.size()) != length - 1) continue;

            for (int gen = 1; gen <= number_gens; ++gen) {
                // always allowed (no reduced‑word condition)
                std::vector<int> t1 = w; t1.push_back(-gen);
                std::vector<int> t2 = w; t2.push_back(gen);
                new_words.push_back(std::move(t1));
                new_words.push_back(std::move(t2));
            }
        }
        words.insert(words.end(),
                     std::make_move_iterator(new_words.begin()),
                     std::make_move_iterator(new_words.end()));
    }

    if (exact_length) {
        // Keep only words of the required length (mirrors Python behaviour)
        words.erase(
            std::remove_if(words.begin(), words.end(),
                [max_length](const std::vector<int>& w){ return static_cast<int>(w.size()) != max_length; }),
            words.end());
    }
}

} // namespace grp


/*
#include "group_word_generator.hpp"
#include <iostream>

int main()
{
    std::vector<std::vector<int>> words;

    // Example 1: reduced words on 2 generators, length ≤ 4
    grp::generate_group_words(words, 2, 4);
    std::cout << "Reduced words (2 generators, ≤4): " << words.size() << '\n';

    // Example 2: all (non‑reduced) words on 3 generators, exact length 3
    grp::generate_all_words(words, 3, 3, true);
    std::cout << "All words (3 generators, length = 3): " << words.size() << '\n';

    // Print a few words (optional)
    for (size_t i = 0; i < std::min(words.size(), size_t{10}); ++i) {
        std::cout << '[';
        for (size_t j = 0; j < words[i].size(); ++j) {
            std::cout << words[i][j];
            if (j + 1 < words[i].size()) std::cout << ',';
        }
        std::cout << "]\n";
    }
}
*/


/* ------------------------------------------------------------------
 *  Public type aliases – they must appear exactly as requested.
 * ------------------------------------------------------------------ */
using Rule  = std::pair<std::vector<int>, std::vector<int>>;   // (lhs , rhs)
using Rules = std::vector<Rule>;

/* ------------------------------------------------------------------
 *  presentation_to_rules
 *
 *  Input  : `relators` – a list of relators, each relator is a word
 *           represented by `std::vector<int>`.
 *
 *  Output : `Rules` – for every relator we create a rule whose left
 *           hand side is the relator itself and whose right hand side
 *           is the empty word (`std::vector<int>{}`).
 *
 *  The function mirrors the Python semantics exactly, but uses the
 *  specified `Rule`/`Rules` aliases.
 * ------------------------------------------------------------------ */
inline Rules presentation_to_rules(const std::vector<std::vector<int>>& relators)
{
    Rules rules;
    rules.reserve(relators.size());                // avoid repeated reallocations

    for (const auto& rel : relators) {
        // right‑hand side = empty word
        rules.emplace_back(rel, std::vector<int>{});
    }
    return rules;                                 // move‑constructed out of the function
}

/* ------------------------------------------------------------------
 *  Overload that accepts an r‑value list of relators.  This allows
 *  callers to pass a temporary without an extra copy.
 * ------------------------------------------------------------------ */
inline Rules presentation_to_rules(std::vector<std::vector<int>>&& relators)
{
    Rules rules;
    rules.reserve(relators.size());

    for (auto& rel : relators) {
        rules.emplace_back(std::move(rel), std::vector<int>{});
    }
    return rules;
}

#endif // GROUP_WORD_GENERATOR_HPP

/*
#include "presentation_to_rules.hpp"
#include <iostream>

int main()
{
    // Example relators (words on 2 generators)
    std::vector<std::vector<int>> relators = {
        { 1, -2, 1 },          // a·b⁻¹·a
        { -1, 2, -1, 2 }       // a⁻¹·b·a⁻¹·b
    };

    // Convert to the required Rule / Rules representation
    Rules rules = presentation_to_rules(relators);

    std::cout << "Created " << rules.size() << " rules:\n";
    for (const auto& r : rules) {
        std::cout << "LHS = [";
        for (size_t i = 0; i < r.first.size(); ++i) {
            std::cout << r.first[i];
            if (i + 1 < r.first.size()) std::cout << ',';
        }
        std::cout << "], RHS = [";
        // RHS is always empty here, but we print it for completeness
        for (size_t i = 0; i < r.second.size(); ++i) {
            std::cout << r.second[i];
            if (i + 1 < r.second.size()) std::cout << ',';
        }
        std::cout << "]\n";
    }
}
*/

Writing word_generator.cpp


In [ ]:
%%writefile detect.cpp
/********************************************************************
 *  free_inclusion.hpp
 *
 *  Translation of the three Python helpers
 *
 *      detect_inverses
 *      detect_duplicates
 *      detect_free_inclusion
 *
 *  Assumptions that are kept from the original Python code
 *  ------------------------------------------------------
 *  * a *word*          → std::vector<int>
 *  * a collection of words → std::vector<std::vector<int>>
 *  * the inverse of a word is obtained by reversing the word and
 *    changing the sign of every generator.
 *
 *  The functions below return the same Boolean result as the Python
 *  versions and use only the standard library.
  ********************************************************************/

#ifndef FREE_INCLUSION_HPP
#define FREE_INCLUSION_HPP

#include <algorithm>   // std::find, std::any_of, std::count
#include <iterator>    // std::distance
#include <vector>

namespace grp {
/* ------------------------------------------------------------------
 *  Helper: inverse of a word.
 *  The inverse is obtained by reversing the word and flipping the sign
 *  of every generator.
 * ------------------------------------------------------------------ */
//inline std::vector<int> group_inverse(const std::vector<int>& w)
//{
//    std::vector<int> inv;
//    inv.reserve(w.size());
//    for (auto it = w.rbegin(); it != w.rend(); ++it)
//        inv.push_back(-(*it));
//    return inv;
//}

/* ------------------------------------------------------------------
 *  Helper: test whether a word is already stored in the container.
 *  (Linear search – identical to Python's `in` operator.)
 * ------------------------------------------------------------------ */
//inline bool contains(const std::vector<std::vector<int>>& container,
//                     const std::vector<int>& word)
//{
//    return std::find(container.begin(), container.end(), word) != container.end();
//}

/* ------------------------------------------------------------------
 *  1. detect_inverses
 *
 *  Returns true iff there exists a word w in *words* such that
 *  the inverse of w also belongs to *words*.
 * ------------------------------------------------------------------ */
inline bool detect_inverses(const std::vector<std::vector<int>>& words)
{
    for (const auto& w : words) {
        if (contains(words, group_inverse(w)))
            return true;
    }
    return false;
}

/* ------------------------------------------------------------------
 *  2. detect_duplicates
 *
 *  Returns true iff at least one word occurs more than once in *words*.
 * ------------------------------------------------------------------ */
inline bool detect_duplicates(const std::vector<std::vector<int>>& words)
{
    for (size_t i = 0; i < words.size(); ++i) {
        // count how many times words[i] appears in the whole container
        if (std::count(words.begin(), words.end(), words[i]) > 1)
            return true;
    }
    return false;
}

/* ------------------------------------------------------------------
 *  3. detect_free_inclusion
 *
 *  The routine mimics the three‑stage Python test:
 *      a) free reduction by cancelling adjacent inverse pairs,
 *      b) cyclic reduction (first = -last),
 *      c) checking all cyclic shifts of the reduced word and of its inverse.
 *
 *  Parameters
 *      word  – the word to test (will be copied, the original is unchanged)
 *      words – the set of words against which we test inclusion
 *
 *  Returns true if the (possibly reduced) word, a cyclic shift of  or a cyclic shift of its inverse is present in *words*.
 * ------------------------------------------------------------------ */
inline bool detect_free_inclusion(std::vector<int> word,
                                  const std::vector<std::vector<int>>& words)
{
    /* --------------------------------------------------------------
     * Stage A : free reduction – cancel i,i+1 when they are inverses.
     * -------------------------------------------------------------- */
    bool cancellation_made = true;
    while (word.size() > 1 && cancellation_made) {
        cancellation_made = false;
        for (size_t i = 0; i + 1 < word.size(); ++i) {
            if (word[i] == -word[i + 1]) {
                // erase the cancelling pair
                word.erase(word.begin() + i, word.begin() + i + 2);
                cancellation_made = true;
                if (word.empty())
                    return true;               // completely cancelled -> free word
                break;                         // restart scanning from the beginning
            }
        }
    }

    /* --------------------------------------------------------------
     * Stage B : cyclic reduction – repeatedly delete the outermost
     *           inverse pair (first, last) while they are opposite.
     * -------------------------------------------------------------- */
    while (word.size() > 1 && word.front() == -word.back()) {
        word.erase(word.begin());                // drop first
        word.pop_back();                         // drop last
        if (word.empty())
            return true;
    }

    /* --------------------------------------------------------------
     * Helper lambda: does any cyclic shift of `w` occur in `words` ?
     * -------------------------------------------------------------- */
    auto shift_in_container = [&](const std::vector<int>& w) -> bool {
        const size_t n = w.size();
        for (size_t shift = 0; shift < n; ++shift) {
            // build the shifted word: w[shift:] + w[:shift]
            std::vector<int> shifted;
            shifted.reserve(n);
            shifted.insert(shifted.end(), w.begin() + shift, w.end());
            shifted.insert(shifted.end(), w.begin(), w.begin() + shift);
            if (contains(words, shifted))
                return true;
        }
        return false;
    };

    /* --------------------------------------------------------------
     * Stage C : check the (reduced) word and the inverse of the word.
     * -------------------------------------------------------------- */
    if (shift_in_container(word))
        return true;

    const std::vector<int> inv = group_inverse(word);
    if (shift_in_container(inv))
        return true;

    return false;
}

/* ------------------------------------------------------------------
 *  Overload that accepts a const reference for the word (makes a copy
 *  internally, exactly as the Python version does).
 * ------------------------------------------------------------------ */
//inline bool detect_free_inclusion(const std::vector<int>& word,
//                                 const std::vector<std::vector<int>>& words)
//{
//    return detect_free_inclusion(std::vector<int>(word), words);
//}

} // namespace grp

#endif // FREE_INCLUSION_HPP

Overwriting detect.cpp


### main.cpp

In [ ]:
%%writefile main.cpp
#include "ordering.cpp"
#include "vector.cpp"
#include "reduce.cpp"
#include "crit_pairs.cpp"
#include "resolve.cpp"
#include "shortcut.cpp"
#include "redundancy.cpp"
#include "knuth-bendix.cpp"
#include "group.cpp"
#include "print.cpp"
#include "word_generator.cpp"
#include "detect.cpp"

// ------------------------------------------------------------------
//  Helper to duplicate a word three times (the Python `word*3`)
// ------------------------------------------------------------------
static std::vector<int> triple_word(const std::vector<int>& w)
{
    std::vector<int> res;
    res.reserve(w.size() * 3);
    res.insert(res.end(), w.begin(), w.end());
    res.insert(res.end(), w.begin(), w.end());
    res.insert(res.end(), w.begin(), w.end());
    return res;
}

static std::vector<int> quadruple_word(const std::vector<int>& w)
{
    std::vector<int> res;
    res.reserve(w.size() * 4);
    res.insert(res.end(), w.begin(), w.end());
    res.insert(res.end(), w.begin(), w.end());
    res.insert(res.end(), w.begin(), w.end());
    res.insert(res.end(), w.begin(), w.end());
    return res;
}


// ------------------------------------------------------------------
//  Main – exact analogue of the Python script
// ------------------------------------------------------------------
int main()
{
    // 1. generate all reduced words on 2 generators of length ≤ 4
    std::vector<std::vector<int>> group_words;
    grp::generate_group_words(
        group_words,
        /*number_gens=*/2,
        /*max_length=*/3,
        /*exact_length=*/false,
        /*remove_cyc_reducible=*/false,
        /*remove_shifts=*/false,
        /*remove_inverses=*/false);

    // If you want the “no‑reduction” variant, just uncomment the call
    // below and comment the one above (exact same arguments, but the three
    // boolean flags set to false).
    /*
    grp::generate_group_words(
        group_words,
        2, 3, false,
        false,   // remove_cyc_reducible
        false,   // remove_shifts
        false);  // remove_inverses
    */

    // 2. build the list `all_starting_relators = [word*3 for word in group_words]`
    std::vector<std::vector<int>> all_starting_relators;
    all_starting_relators.reserve(group_words.size());
    for (const auto& w : group_words) {
        all_starting_relators.push_back(quadruple_word(w));
    }

    // 3. turn each relator into a rule (rhs = empty word)
    Rules all_starting_rules = presentation_to_rules(all_starting_relators);

    // 4. copy the rules into the mutable container `rules`
    Rules rules = all_starting_rules;   // simple copy (Rule is trivially copyable)

    // 5. finally add the free‑group rules
    add_free_group_rules(rules);

    /*
    // ------------------------------------------------------------------
    // optional: show how many rules we ended up with
    // ------------------------------------------------------------------
    std::cout << "Number of generated rules: " << rules.size() << '\n';

    // (you can also print a few rules if you like)
    for (size_t i = 0; i < std::min(rules.size(), size_t{16}); ++i) {
        const auto& lhs = rules[i].first;
        const auto& rhs = rules[i].second;   // always empty in this stage
        std::cout << "Rule " << i << ":  LHS = [";
        for (size_t j = 0; j < lhs.size(); ++j) {
            std::cout << lhs[j];
            if (j + 1 < lhs.size()) std::cout << ',';
        }
        std::cout << "]  RHS = [";
        for (size_t j = 0; j < rhs.size(); ++j) {
            std::cout << rhs[j];
            if (j + 1 < rhs.size()) std::cout << ',';
        }
        std::cout << "]\n";
    }
    */

    /*
    print("B(2,3). Starting rules:", rules)
    print("Trying shortlex:")
    rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
    if rounds != -1:
      print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
    else:
      print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
    print(rules)
    print("Confluence:", check_confluence(rules,print_progress=False))
    print("---------")
    */

    std::cout << "B(2,4). Number of starting rules:\n" << rules.size() << '\n';
    std::cout << "Trying RPO pos first:\n";

    int rounds = knuth_bendix(rules, recursive_path_order_with_positive_first, true, 4, false,true,false);
    if(rounds != -1)
      std::cout << "Finished in " << rounds << " rounds.\nTotal number of rules:" << rules.size() << "\n";
    else
      std::cout << "Terminated by reaching max_rounds.\nTotal number of rules:" << rules.size() << "\n";

    std::cout << "Confluence:" << check_confluence(rules) << "\n"; // Pass the non-const object
    std::cout << "---------\n";

    return 0;
}

Overwriting main.cpp


In [ ]:
!g++ main.cpp -o main
!./main

B(2,4). Number of starting rules:
56
Trying RPO pos first:
Round 1
Redundancies eliminated. 40 rules total.
Crit pairs: 448
New rules added. 95 rules total.
95 rules total at the end of round 1
Round 2
Shortcuts made. 95 rules total.
Redundancies eliminated. 19 rules total.
Crit pairs: 249
New rules added. 166 rules total.
166 rules total at the end of round 2
Round 3
Shortcuts made. 166 rules total.
Redundancies eliminated. 39 rules total.
Crit pairs: 1022
New rules added. 735 rules total.
735 rules total at the end of round 3
Round 4
Shortcuts made. 735 rules total.
Redundancies eliminated. 263 rules total.
Crit pairs: 49271
^C


## Python Knuth-Bendix implementation

In [ ]:
from tqdm import tqdm

In [ ]:
rules = []
''' rules stored as [length of LHS, LHS, RHS] '''
''' for example: '''
''' 1234->567 is stored as [4, 1,2,3,4, 5,6,7] '''
''' x_1 x_2 x_1^{-1} -> x_2^2 stored as [3, 1,2,-1, 2,2] '''
''' together: rules = [ [4, 1,2,3,4, 5,6,7] , [3, 1,2,-1, 2,2]] '''

def rule_left(rule):
  '''extracts lhs of rule'''
  return rule[1:rule[0]+1]

def rule_right(rule):
  '''extracts rhs of rule'''
  return rule[rule[0]+1:]


def alphabet_order(a,b):
  ''' x_1 < x_1^{-1} < x_2 < x_2^{-1} < .... '''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  if abs(a) > abs(b):
    return 1
  elif abs(a) < abs(b):
    return -1
  elif a<b:
    return -1
  elif a>b:
    return 1
  else:
    return 0

def alphabet_order_positive_first(a,b):
  ''' x_1 < x_2 < ... < x_n < x_1^{-1} < x_2^{-1} < ... < x_n^{-1} '''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  if a > 0 and b < 0:
    return -1
  if a < 0 and b > 0:
    return 1
  if a > 0 and b > 0:
    if a > b:
      return 1
    elif a < b:
      return -1
    else:
      return 0
  if a < 0 and b < 0:
    if a < b:
      return 1
    elif a > b:
      return -1
    else:
      return 0



def chenadec_order(a,b):
  ''' x_2p > x_2p^{-1} > x_{2p-2} > .... > x_2^{-1} > x_1 > x_1^{-1} > ... > x_{2p-1}^{-1}'''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  if a%2 == 0 and b%2 == 1:
    return 1
  elif a%2 == 1 and b%2 == 0:
    return -1
  elif a%2 == 0 and b%2 == 0:
    if abs(a) > abs(b):
      return 1
    elif abs(a) < abs(b):
      return -1
    elif a > b:
      return 1
    elif a < b:
      return -1
    else:
      return 0
  elif a%2 == 1 and b%2 == 1:
    if abs(a) < abs(b):
      return 1
    elif abs(a) > abs(b):
      return -1
    elif a > b:
      return 1
    elif a < b:
      return -1
    else:
      return 0
  return 0

def shortlex(word1,word2,alphabet_order=alphabet_order):
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  if len(word1) > len(word2):
    return 1
  elif len(word1) < len(word2):
    return -1
  else:
    for i in range(len(word1)):
      if alphabet_order(word1[i],word2[i]) == 1:
        return 1
      elif alphabet_order(word1[i],word2[i]) == -1:
        return -1
    return 0

def shortlex_with_chenadec(a,b):
  return shortlex(a,b,alphabet_order=chenadec_order)

def shortlex_with_positive_first(a,b):
  return shortlex(a,b,alphabet_order=alphabet_order_positive_first)

def lex(word1,word2,alphabet_order=alphabet_order):
  ''' may lead to infinite reduction chain, do not use unless you know what you are doing '''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  for i in range(min(len(word1),len(word2))):
    if alphabet_order(word1[i],word2[i]) == 1:
      return 1
    elif alphabet_order(word1[i],word2[i]) == -1:
      return -1
  if len(word1) > len(word2):
    return 1
  elif len(word1) < len(word2):
    return -1
  # else:
  return 0

def recursive_path_order(word1,word2,alphabet_order=alphabet_order, use_dynprog = True):
  ''' as in Hermiller-Shapiro paper '''
  ''' 1 means LHS > RHS, -1 means LHS < RHS, 0 means LHS = RHS '''
  ''' by default uses naive dynamic programming implementation it's 10-30% faster on words up 200 letters '''
  if use_dynprog and (len(word1) > 20 or len(word2) > 20):
    return recursive_path_order_dyn_prog(word1,word2,alphabet_order=alphabet_order)

  if len(word1) == 1 and len(word2) == 1:
    return alphabet_order(word1[0],word2[0])
  if len(word1) > 0 and len(word2) == 0:
      return 1
  if len(word1) == 0 and len(word2) > 0:
      return -1
  if word1[0] == word2[0]:
    # print("case 1")
    return recursive_path_order(word1[1:],word2[1:],alphabet_order)
  if alphabet_order(word1[0],word2[0]) == 1 and recursive_path_order(word1,word2[1:],alphabet_order) == 1:
    # print("case 2")
    return 1
  # print("case 3")
  ord = recursive_path_order(word1[1:],word2,alphabet_order)
  if ord == 1 or ord == 0:
    return 1
  return -1

def recursive_path_order_dyn_prog(word1, word2, alphabet_order=alphabet_order):
  """Iterative implementation of the recursive path order.

  Args:
    word1: The first word.
    word2: The second word.
    alphabet_order: The alphabet order function.

  Returns:
    1 if word1 > word2, -1 if word1 < word2, 0 if word1 == word2.
  """
  if len(word1) == 1 and len(word2) == 1:
    return alphabet_order(word1[0],word2[0])
  if len(word1) > 0 and len(word2) == 0:
      return 1
  if len(word1) == 0 and len(word2) > 0:
      return -1

  dynamic_prog_table = [[0 for _ in range(len(word2)+1)] for _ in range(len(word1)+1)]

  len1=len(word1)
  len2=len(word2)
  for k in range(len2):
    dynamic_prog_table[len1][k] = -1
  for k in range(len1):
    dynamic_prog_table[k][len2] = 1
    dynamic_prog_table[len1][len2] = 0
  # for ii in range(len1+1):
    # print(dynamic_prog_table[len1-ii])
  # print("-------")
  for diag in reversed(range(len1+len2-1)): # i+j=diag
    # print("Diag =",diag)
    start_i = 0 if diag-len2+1 < 0 else diag-len2+1
    end_j = 0 if diag-len1+1 < 0 else diag-len1+1
    for i in range(start_i, diag-end_j+1):
      j = diag-i
      if word1[i] == word2[j]:
        dynamic_prog_table[i][j] = dynamic_prog_table[i+1][j+1]
      elif alphabet_order(word1[i], word2[j]) == 1 and dynamic_prog_table[i][j+1] == 1:
        dynamic_prog_table[i][j] = 1
      # elif word1[i] < word2[j] and dynamic_prog_table[i+1][j] == -1:
        # dynamic_prog_table[i][j] = -1
      elif dynamic_prog_table[i+1][j] >= 0:
        dynamic_prog_table[i][j] = 1
      # elif dynamic_prog_table[i][j+1] >= 0:
        # dynamic_prog_table[i][j] = -1
      else:
        dynamic_prog_table[i][j] = -1
      # for ii in range(len1+1):
        # print(dynamic_prog_table[len1-ii])
      # print("-------")
  return dynamic_prog_table[0][0]



def recursive_path_order_with_chenadec(a,b):
  return recursive_path_order(a,b,alphabet_order=chenadec_order)

def recursive_path_order_with_positive_first(a,b):
  return recursive_path_order(a,b,alphabet_order=alphabet_order_positive_first)

def wreath_order(word1,word2,alphabet_order=alphabet_order):
  ''' not implemented '''
  return 0

def reorder_rules(rules, order):
  ''' for each rule u->v, rewrite it as u->v or v->u according to order '''
  for i in range(len(rules)):
    if order(rule_left(rules[i]),rule_right(rules[i])) == -1:
      rules[i] = [len(rule_right(rules[i]))] + rule_right(rules[i]) + rule_left(rules[i])

# shortlex([1,2],[1,2,1],alphabet_order)


In [ ]:
def apply_rule(word, rule, place=0):
  ''' stupidly searches for lhs of rule starting from place and replaces with rhs '''
  ''' there are faster ways to implement this but this at least works '''
  ''' returns new word, or unchanged word if rule does not apply '''
  lhs = rule_left(rule)
  rhs = rule_right(rule)
  for i in range(place,len(word) - len(lhs) + 1):
    if word[i : i + len(lhs)] == lhs:
      word = word[:i] + rhs + word[i+len(lhs):]
  return word

#apply_rule([1,2,3,4,5],[2,3,4,-1,-1,-1])

def reduce(word, rules, skip_rule = -1, print_progress = False):
  ''' reduces the word, choosing first applicable place of first applicable rule '''
  ''' skip_rule is the rule to skip, needed for redundancy check in K-B algorithm '''
  ''' returns reduces word '''
  while True:
    if print_progress:
      print("new round...")
    starting_word = word
    for i in range(len(rules)):
      if i != skip_rule:
        word = apply_rule(word, rules[i])
    if word == starting_word:
      break
  return word


In [ ]:
crit_pairs = []
# for overlapping rules u_1 v -> w_1, v u_2 -> w_2, crit pair is the result of applying each rule to u_1 v u_2:
# [w_1 u_2, u_1 w_2]
# for inculsive rules v -> w_1, u v u' -> w_2, crit pair is the result of applying each rule to u v u':
# [w_2, u 2_1 u']
# (u, u' allowed to be empty in the code below, u_1, u_2 shouldn't be empty unless I messed up)


def make_crit_pairs(rules, crit_pairs, reduce_immediately = True, start_at_rule=0, max_crit_length=0, reset_pairs = True, \
                    print_progress=False, print_progress_percentage=False):
  ''' returns list of crit pairs for the given list of rules'''
  ''' if reduce_immediately == True, the crit pairs resolved by existing rules are skipped '''
  ''' if start_at_rule > 0, only crit pairs involving at least one rule with number >= start_at_rule will be processed '''
  ''' '''
  if reset_pairs:
    crit_pairs.clear()

  #lhs overlap
  if print_progress_percentage:
    print("Building crit pairs - overlap:")
  i_range = tqdm(range(len(rules))) if print_progress_percentage else range(len(rules))
  for i in i_range:
    for j in range(len(rules)):
      if i < start_at_rule and j < start_at_rule:
        continue
      lhs1 = rule_left(rules[i])
      lhs2 = rule_left(rules[j])
      #if print_progress:
      #  print(lhs1,lhs2)
      for k in range(1,min(len(lhs2),len(lhs1))): # checking for overlap of size k
        if (max_crit_length==0 or len(lhs1)+len(lhs2)-k <= max_crit_length) and lhs1[-k:] == lhs2[:k]:
          if reduce_immediately:
            reduced_word1 = reduce(rule_right(rules[i])+lhs2[k:], rules)
            reduced_word2 = reduce(lhs1[:-k]+rule_right(rules[j]), rules)
            if print_progress:
                print("Overlap:", lhs1[:-k] + lhs1[-k:] + lhs2[k:], "with lhs:", lhs1, lhs2, "Reduced words:", reduced_word1, reduced_word2)
            if reduced_word1 != reduced_word2:
              crit_pairs.append([reduced_word1, reduced_word2])
          else:
            if print_progress:
              print("Overlap:", lhs1[:-k] + lhs1[-k:] + lhs2[k:], "with lhs:", lhs1, lhs2, "Words:", rule_right(rules[i])+lhs2[k:], lhs1[:-k]+rule_right(rules[j]))
            crit_pairs.append([rule_right(rules[i])+lhs2[k:] , lhs1[:-k]+rule_right(rules[j])])

  #lhs inclusion
  if print_progress_percentage:
    print("Building crit pairs - inclusion:")
  i_range = tqdm(range(len(rules))) if print_progress_percentage else range(len(rules))
  for i in i_range:
    lhs1 = rule_left(rules[i])
    for j in range(start_at_rule,len(rules)):
      if i == j:
        continue
      lhs2 = rule_left(rules[j])
      if (max_crit_length==0 or len(lhs2) <= max_crit_length) and len(lhs2) >= len(lhs1):
        for k in range(len(lhs2)-len(lhs1)+1):
          if lhs2[k:k+len(lhs1)] == lhs1:
            if reduce_immediately:
              reduced_word1 = reduce(lhs2[:k]+rule_right(rules[i])+lhs2[k+len(lhs1):], rules)
              reduced_word2 = reduce(rule_right(rules[j]), rules)
              if print_progress:
                print("Inclusion:", lhs2, "with lhs:", lhs1, lhs2, "Reduced words:", reduced_word1, reduced_word2)
              if reduced_word1 != reduced_word2:
                crit_pairs.append([reduced_word1, reduced_word2])
            else:
              if print_progress:
                print("Inclusion:", lhs2, "with lhs:", lhs1, lhs2, "Words:", lhs2[:k]+rule_right(rules[i])+lhs2[k+len(lhs1):], rule_right(rules[j]))
              crit_pairs.append([lhs2[:k]+rule_right(rules[i])+lhs2[k+len(lhs1):], rule_right(rules[j])])
      elif (i < start_at_rule and j >= start_at_rule) and (max_crit_length==0 or len(lhs1) <= max_crit_length) and len(lhs2) < len(lhs1):
        # only pairs (i,j) where i < start_at_rule and j >= start_at_rule need to be swapped
        # pairs where both i,j >= start_at_rule will be encountered as (i,j) and as (j,i) anyway
        for k in range(len(lhs1)-len(lhs2)+1):
          if lhs1[k:k+len(lhs2)] == lhs2:
            if reduce_immediately:
              reduced_word1 = reduce(lhs1[:k]+rule_right(rules[j])+lhs1[k+len(lhs2):], rules)
              reduced_word2 = reduce(rule_right(rules[i]), rules)
              if print_progress:
                print("Inclusion:", lhs1, "with lhs:", lhs2, lhs1, "Reduced words:", reduced_word1, reduced_word2)
              if reduced_word1 != reduced_word2:
                crit_pairs.append([reduced_word1, reduced_word2])
            else:
              if print_progress:
                print("Inclusion:", lhs1, "with lhs:", lhs2, lhs1, "Words:", lhs1[:k]+rule_right(rules[j])+lhs1[k+len(lhs2):], rule_right(rules[i]))
              crit_pairs.append([lhs1[:k]+rule_right(rules[j])+lhs1[k+len(lhs2):], rule_right(rules[i])])
  return crit_pairs

def check_confluence(rules,crit_pairs=[], max_crit_length = 0, erase_pair_list = True, print_progress = False):
  ''' checks if the rewriting system is confluent'''
  ''' redundant if knuth_bendix() converged '''
  ''' will clear list of crit pairs after running if erase_pair_list == True (default) '''
  if crit_pairs == []:
    crit_pairs = make_crit_pairs(rules, crit_pairs, max_crit_length=max_crit_length,print_progress=print_progress)
  for i in range(len(crit_pairs)):
    if print_progress:
      print(crit_pairs[i])
    word1 = reduce(crit_pairs[i][0],rules, print_progress=False)
    word2 = reduce(crit_pairs[i][1],rules, print_progress=False)
    if word1 != word2:
      if print_progress:
        print("Confluence fail at", crit_pairs[i][0], crit_pairs[i][1], "-->", word1, word2)
      if erase_pair_list:
        crit_pairs.clear()
      return False
  if erase_pair_list:
        crit_pairs.clear()
  return True


In [ ]:
def resolve_crit_pair(rules, crit_pairs, i, order=shortlex, print_progress = False):
  ''' resolves crit pair number i '''
  ''' reduces both words in crit pair i using existing rules '''
  ''' if results are not equal, adds a rule word1 -> word2 or word2 -> word1 depending on order '''
  ''' reduction is redundant if crit_pairs was obtained from make_crit_pairs() with reduce_immediately = True '''
  if print_progress:
    print("Resolving ", crit_pairs[i])
  word1 = reduce(crit_pairs[i][0],rules, print_progress=print_progress)
  word2 = reduce(crit_pairs[i][1],rules, print_progress=print_progress)
  if word1 != word2:
    if order(word1,word2) == 1:
      rules.append([len(word1)] + word1 + word2)
    else:
      rules.append([len(word2)] + word2 + word1)
    crit_pairs.remove(crit_pairs[i])
    if print_progress:
      print("Added rule", rules[-1])
    return 1
  crit_pairs.remove(crit_pairs[i])
  if print_progress:
      print("No rule added")
  return 0

def resolve_all_crit_pairs(rules, crit_pairs = [], order=shortlex, reduce_crit_immediately = True, go_from_the_end=True, print_progress = False, print_crit_pairs_progress = False):
  ''' Resolves crit_pairs going from the end or the beginning of the list according to go_from_the_end'''
  ''' Returns True if any new rules were added '''
  if crit_pairs == []:
    crit_pairs = make_crit_pairs(rules, crit_pairs, reduce_immediately=reduce_crit_immediately, print_progress=print_crit_pairs_progress)
  rule_counter = 0
  if print_progress:
    print("Resolving crit pairs...")
    pbar = tqdm(total = len(crit_pairs))
  while len(crit_pairs) > 0:
    rule_number_to_resolve = -1 if go_from_the_end else 0
    rule_counter+=resolve_crit_pair(rules, crit_pairs, rule_number_to_resolve, order=order)
    if print_progress:
        # print("Crit pairs left to resolve:", len(crit_pairs), "     ", end="\r")
        pbar.update(1)
  if print_progress:
      # print("                                                           ", end="\r")
      pbar.close()

  if rule_counter > 0:
    if print_progress:
      print("Added", rule_counter, "new rules")
    return True
  return False

In [ ]:
def shortcut(rules, rule_number):
  ''' Reduces RHS of rule number rule_number '''
  ''' Returns True if a shortcut was made '''
  rhs=rule_right(rules[rule_number])
  lhs=rule_left(rules[rule_number])
  reduced_rhs = reduce(rhs,rules)
  if reduced_rhs != rhs:
   # rules[rule_number] = [len(lhs)] + lhs + reduced_rhs
    rules[rule_number][len(lhs)+1:] = reduced_rhs
    return True
  return False

def make_all_shortcuts(rules):
  ''' Reduces RHS of all rules '''
  ''' Returns True if any shortcuts were made '''
  changes_made = False
  for i in range(len(rules)):
    changes_made = shortcut(rules,i) or changes_made
  return changes_made


In [ ]:
def is_rule_redundant(rules, rule_number):
  ''' Checks if rule number rule_number is redundant '''
  ''' by reducing its LHS and RHS by all other rules '''
  ''' Returns True if rule is redundant '''
  if reduce(rule_left(rules[rule_number]), rules, skip_rule=rule_number) == \
     reduce(rule_right(rules[rule_number]), rules, skip_rule=rule_number):
    return True
  return False

def eliminate_redundancy(rules,start_at_rule=0, go_from_the_end=True):
  ''' Eliminates redundant rules '''
  ''' Returns True if any rules were eliminated and '''
  ''' the new number of first new rule to use when making crit pairs '''
  redundancies_were_present = False
  i=0
  new_first_rule=start_at_rule
  if go_from_the_end:
    while i < len(rules):
      # each iteration of cycle either i increases by 1, or len(rules) decreases by 1 '''
      if is_rule_redundant(rules,len(rules)-1-i):
        redundancies_were_present = True
        if len(rules)-1-i < new_first_rule:
          new_first_rule-=1 # old rule was deleted, need to shift first new rule number
        del rules[-1-i]
      else:
        i+=1
  else:
    while i < len(rules):
      # each iteration of cycle either i increases by 1, or len(rules) decreases by 1 '''
      if is_rule_redundant(rules,i):
        redundancies_were_present = True
        rules.remove(rules[i])
        if i < new_first_rule:
          new_first_rule-=1 # old rule was deleted, need to shift first new rule number
      else:
        i+=1
  return redundancies_were_present, new_first_rule

In [ ]:
def knuth_bendix(rules, order=shortlex, reduce_crit_immediately = True, max_rounds=-1, recheck_old_rules = False, \
                 print_progress = False, print_crit_pairs_progress = False):
  ''' Runs Knuth-Bendix algorithm on the given list of rules '''
  ''' Follows Algorithm 6.2.11 in Word Processing in Groups by Epstein at al. (Section 6.2) '''
  ''' Returns -1 if terminated by reaching max_rounds; or number of rounds if terminated by attaining confluence '''
  changes_made = True
  rounds=0
  first_new_rule=0
  crit_pairs = []
  while changes_made:
    rounds=rounds+1
    if max_rounds >= 0 and rounds > max_rounds:
      print("Max rounds reached:", max_rounds)
      print("Making shortcuts and removing redundancies, then stopping.")
      # return -1
    if print_progress:
      print("Round", rounds)
    changes_made = False
    shortcuts_made = False
    new_rules_added = False
    redundancies_eliminated = False

    # make all shortcuts
    shortcuts_made = make_all_shortcuts(rules)
    if print_progress and shortcuts_made:
      print("Shortcuts made.", len(rules), "rules total.")

    # remove redundant rules
    redundancies_eliminated, first_new_rule = eliminate_redundancy(rules, first_new_rule)
    if print_progress and redundancies_eliminated:
      print("Redundancies eliminated.", len(rules), "rules total.")
      # print(len(rules), "rules total:\n", rules)

    if max_rounds >= 0 and rounds > max_rounds:
      print("Stopping because max rounds reached:", max_rounds)
      return -1

    # make crit pairs only using at least one new rule
    assert(crit_pairs == [])
    if recheck_old_rules:
      make_crit_pairs(rules, crit_pairs, reduce_immediately=reduce_crit_immediately, start_at_rule=0, \
                      print_progress=print_crit_pairs_progress, print_progress_percentage = print_progress)
    else:
      make_crit_pairs(rules, crit_pairs, reduce_immediately=reduce_crit_immediately, start_at_rule=first_new_rule, \
                      print_progress=print_crit_pairs_progress, print_progress_percentage = print_progress)

    if print_progress:
      print("Crit pairs:", len(crit_pairs))

    # all rules become old
    first_new_rule = len(rules)

    # add new rules to resolve all crit pairs
    new_rules_added = resolve_all_crit_pairs(rules, crit_pairs=crit_pairs, order=order, reduce_crit_immediately=reduce_crit_immediately, \
                                             print_progress=print_progress)
    if print_progress and new_rules_added:
      print("New rules added.", len(rules), "rules total.")
    changes_made = shortcuts_made or redundancies_eliminated or new_rules_added
    if print_progress:
      # print(len(rules), "rules total:\n", rules)
        print(len(rules), "rules total at the end of round", rounds)
  return rounds

In [ ]:
def group_inverse(word):
  neg_word = [-x for x in word]
  return neg_word[::-1]

def symmetrize_group_rule(rules,i,order=shortlex):
  ''' symmetrizes rule number i assuming group structure: '''
  ''' for a rule u=v, it computes r=uv^{-1} and iterates over all cyclic shifts and their inverses r' of r: '''
  ''' for each r', it adds all rules obtained from writing r'as u' = v' in all possible ways '''
  ''' rule of length n adds 2n*n rules '''
  word = rule_left(rules[i]) + group_inverse(rule_right(rules[i]))
  # print(word)

  for j in range(len(word)):
    shifted_word = word[j:] + word[:j]
    # print("shifted word = ", shifted_word)
    for i in range(len(word)):
      word1 = shifted_word[:i]
      word2 = group_inverse(shifted_word[i:])
      if order(word1,word2) == 1:
        rules.append([len(word1)] + word1 + word2)
        # print("added rule:", [len(word1)] + word1 + word2)
      elif order(word1,word2) == -1:
        rules.append([len(word2)] + word2 + word1)
        # print("added rule:", [len(word2)] + word2 + word1)
      # else:
        # print("error")

  # print(len(rules), "before inverse:", rules)
  word = group_inverse(word)
  # print("inverse word:", word)
  for j in range(len(word)):
    shifted_word = word[j:] + word[:j]
    for i in range(len(word)):
      word1 = shifted_word[:i]
      word2 = group_inverse(shifted_word[i:])
      if order(word1,word2) == 1:
        rules.append([len(word1)] + word1 + word2)
      elif order(word1,word2) == -1:
        rules.append([len(word2)] + word2 + word1)
      # else:
        # print("error")

def add_free_group_rules(rules):
  ''' adds rules of the form x x^{-1} -> 1, x^{-1} x -> 1 for all x as high as encountered in rules '''
  max_x = 1
  for i in range(len(rules)):
    max_tmp = max(rules[i][1:])
    min_tmp = min(rules[i][1:])
    if max_tmp > max_x:
      max_x = max_tmp
    if -min_tmp > max_x:
      max_x = -min_tmp
  for i in reversed(range(1,max_x+1)):
    rules.insert(0,[2,-i,i])
    rules.insert(0,[2,i,-i])

### Examples

In [ ]:
# Abelian group on a, b
rules = [[4,1,2,-1,-2]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, recheck_old_rules = True, print_progress=False, print_crit_pairs_progress = False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b
rules = [[4,1,2,-1,-2]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, recheck_old_rules = False, print_progress=False, print_crit_pairs_progress = False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")


Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [4, 1, 2, -1, -2]]
Finished in 6 rounds.
Total number of rules: 8
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 2, -1, -1, 2], [2, 2, 1, 1, 2], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2]]
Confluence: True
-------
Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [4, 1, 2, -1, -2]]
Finished in 6 rounds.
Total number of rules: 8
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 2, -1, -1, 2], [2, 2, 1, 1, 2], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2]]
Confluence: True
-------


In [ ]:
# Abelian group on a, b
rules = [[4,1,2,-1,-2]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b but the rule is not order-reducing:
rules = [[2,1,2,2,1]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b but the rule is not order-reducing
# but we reorder rules according to shortlex first lol:
rules = [[2,1,2,2,1]] #Z^2
add_free_group_rules(rules)
reorder_rules(rules,shortlex)
print("Starting rules after reordering:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b but the rule is order-increasing
# but we reorder rules according to shortlex first lol:
rules = [[1,1,2,1,-2]] #Z^2
add_free_group_rules(rules)
reorder_rules(rules,shortlex)
print("Starting rules after reordering:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b but the rule is written as aba^{-1}=b:
rules = [[3,1,2,-1,2]] #Z^2
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# Abelian group on a, b, c
rules = [[4,1,2,-1,-2], [4,1,3,-1,-3], [4,2,3,-2,-3]] #Z^3
add_free_group_rules(rules)
print("Starting rules:", rules)
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=20, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("Reduction example:")
print([-3,-1,-2,3,2,1,1,1,3,2,2,-1,-3,-3], "->", reduce([-3,-1,-2,3,2,1,1,1,3,2,2,-1,-3,-3],rules))
print("-------")


Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [4, 1, 2, -1, -2]]
Finished in 6 rounds.
Total number of rules: 8
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 2, -1, -1, 2], [2, 2, 1, 1, 2], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2]]
Confluence: True
-------
Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 1, 2, 2, 1]]
Max rounds reached: 20
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 20
Terminated by reaching max_rounds.
Total number of rules: 48
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 1, 2, 2, 1], [3, 2, 1, -2, 1], [3, -1, 2, 1, 2], [4, -1, 2, 2, 1, 2, 2], [2, 2, -1, -1, 2], [4, 2, 1, 1, -2, 1, 1], [2, -2, 1, 1, -2], [5, 2, 1, 1, 1, -2, 1, 1, 1], [5, -1, 2, 2, 2, 1, 2, 2, 2], [2, -2, -1, -1, -2], [6, -1, 2, 2, 2, 2, 1, 2, 2, 2, 2], [6, 2, 1, 1, 1, 1, -2, 1, 1, 1, 1], [7, 2, 1, 1, 1, 1, 1, -2, 1, 1, 1, 1, 1], [7, -1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2], [8, -1, 2, 2, 2, 2, 2, 2, 1, 

In [ ]:
# RAAG group on a, b, c with [a,b]=[b,c]=1
rules = [[4,1,2,-1,-2],[4,2,3,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

rules = [[4,1,2,-1,-2],[4,2,3,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

rules = [[4,1,2,-1,-2],[4,2,3,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")




Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [4, 1, 2, -1, -2], [4, 2, 3, -2, -3]]
Trying shortlex:
Max rounds reached: 10
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 10
Terminated by reaching max_rounds.
Total number of rules: 73
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 2, -1, -1, 2], [2, 3, -2, -2, 3], [2, 2, 1, 1, 2], [2, 3, 2, 2, 3], [2, -3, -2, -2, -3], [2, -2, -1, -1, -2], [3, -3, -1, -2, -2, -3, -1], [3, 3, 1, 2, 2, 3, 1], [3, 3, -1, 2, 2, 3, -1], [3, 3, -1, -2, -2, 3, -1], [2, -2, 1, 1, -2], [2, -3, 2, 2, -3], [4, 3, -1, -1, -2, -2, 3, -1, -1], [4, 3, -1, -1, 2, 2, 3, -1, -1], [4, 3, 1, 1, 2, 2, 3, 1, 1], [4, -3, -1, -1, -2, -2, -3, -1, -1], [5, -3, -1, -1, -1, -2, -2, -3, -1, -1, -1], [3, -3, 1, -2, -2, -3, 1], [5, 3, 1, 1, 1, 2, 2, 3, 1, 1, 1], [5, 3, -1, -1, -1, 2, 2, 3, -1, -1, -1], [5, 3, -1, -1, -1, -2, -2, 3, -1, -1, -1], [3, -3, 1, 2, 2, -3

In [ ]:
# RAAG group on a, b, c with [a,b]=[a,c]=1
rules = [[4,1,2,-1,-2],[4,1,3,-1,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

rules = [[4,1,2,-1,-2],[4,1,3,-1,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

rules = [[4,1,2,-1,-2],[4,1,3,-1,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=10, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [4, 1, 2, -1, -2], [4, 1, 3, -1, -3]]
Trying shortlex:
Finished in 6 rounds.
Total number of rules: 14
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 2, -1, -1, 2], [2, 3, -1, -1, 3], [2, 2, 1, 1, 2], [2, 3, 1, 1, 3], [2, -3, -1, -1, -3], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2], [2, -3, 1, 1, -3]]
Confluence: True
-------
Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [4, 1, 2, -1, -2], [4, 1, 3, -1, -3]]
Trying recursive path ordering:
Finished in 6 rounds.
Total number of rules: 14
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 2, -1, -1, 2], [2, 3, -1, -1, 3], [2, 2, 1, 1, 2], [2, 3, 1, 1, 3], [2, -3, -1, -1, -3], [2, -2, -1, -1, -2], [2, -2, 1, 1, -2], [2, -3, 1, 1, -3]]
Confluence: True
-------
Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [4, 1, 2, -1

In [ ]:
# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with positive first order:")
rounds = knuth_bendix(rules, order=shortlex_with_positive_first, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering with Chenadec order:")
rounds = knuth_bendix(rules, order=recursive_path_order_with_chenadec, reduce_crit_immediately=True, max_rounds=7, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")



# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
symmetrize_group_rule(rules,0)
symmetrize_group_rule(rules,1)
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex on symmetrized rules:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=5, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
symmetrize_group_rule(rules,0)
symmetrize_group_rule(rules,1)
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering on symmetrized rules:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=5, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("-------")

# braid group < a,b,c | a^3 = b^2 = c >
rules = [[3,1,1,1,2,2], [3,1,1,1,3]]
symmetrize_group_rule(rules,0)
symmetrize_group_rule(rules,1)
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order on symmetrized rules:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=5, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))





Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [3, 1, 1, 1, 2, 2], [3, 1, 1, 1, 3]]
Trying shortlex:
Max rounds reached: 7
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 7
Terminated by reaching max_rounds.
Total number of rules: 114
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 2, 2, 3], [2, 3, 1, 1, 3], [2, 3, -1, -1, 3], [2, 1, 1, -1, 3], [3, -1, -1, 3, 1], [2, 3, 2, 2, 3], [2, 3, -2, 2], [2, -2, 3, 2], [3, -2, -1, 3, 2, -1], [3, -2, 1, 3, 2, 1], [2, 2, -3, -2], [2, -3, 1, -1, -1], [2, 1, -3, -1, -1], [2, -3, -1, -1, -3], [3, -1, -1, 2, 1, -2], [2, -3, 2, -2], [2, -3, -2, -2, -3], [2, -2, -2, -3], [3, 1, -2, -3, -1, -1, -2], [3, -1, -1, -1, -3], [3, 2, -1, -3, -2, -1], [3, 2, -1, -1, -2, 1], [3, 2, 1, -2, -2, 1, 2], [4, -2, 1, 2, 3, 2, 1, 2], [3, 2, -1, -2, -2, -1, 2], [4, -2, -1, 2, 3, 2, -1, 2], [5, -2, -1, 2, -1, 3, 2, -1, 2, -1], [5, -2, -1, 2, 1, 3, 2, -1, 2

In [ ]:
# genus 2 orientable surface:
rules = [[8,1,2,3,4,-1,-2,-3,-4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=12, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,2,3,4,-1,-2,-3,-4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=12, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,2,3,4,-1,-2,-3,-4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=12, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))


Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 4, -4], [2, -4, 4], [8, 1, 2, 3, 4, -1, -2, -3, -4]]
Trying shortlex:
Max rounds reached: 12
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 12
Terminated by reaching max_rounds.
Total number of rules: 44
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 4, -4], [2, -4, 4], [4, 4, -1, -2, -3, -3, -2, -1, 4], [4, 2, 3, 4, -1, -1, 4, 3, 2], [4, 4, 3, 2, 1, 1, 2, 3, 4], [4, 3, 4, -1, -2, -2, -1, 4, 3], [4, -4, -3, -2, -1, -1, -2, -3, -4], [5, -2, -1, 4, 3, 2, 3, 4, -1], [5, -3, -2, -1, 4, 3, 4, -1, -2], [4, -4, 1, 2, 3, 3, 2, 1, -4], [8, -3, -2, -1, 4, -2, -1, 4, 3, 4, -1, -2, 4, -1, -2], [8, -2, -1, 4, 3, -1, 4, 3, 2, 3, 4, -1, 3, 4, -1], [4, -2, -3, -4, 1, 1, -4, -3, -2], [11, -2, -1, 4, 3, -1, 4, 3, -1, 4, 3, 2, 3, 4, -1, 3, 4, -1, 3, 4, -1], [11, -3, -2, -1, 4, -2, -1, 4, -2, -1, 4, 3, 4, -1, -2, 4, -1, -2, 4, -1, -2], 

In [ ]:
# non-orientable surface group
rules = [[8,1,1,2,2,3,3,4,4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=15, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,1,2,2,3,3,4,4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with Chenadec alphabet order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=15, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,1,2,2,3,3,4,4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying shortlex with positive first order:")
rounds = knuth_bendix(rules, order=shortlex_with_positive_first, reduce_crit_immediately=True, max_rounds=15, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

rules = [[8,1,1,2,2,3,3,4,4]]
# rules = [[2, 1, -1, ], [2,-1,1,], [2,2,-2,], [2,-2,2,], [2,3,-3,], [2,-3,3,], [6,1,2,3,-1,-2,-3]]
add_free_group_rules(rules)
print("Starting rules:", rules)
print("Trying recursive path ordering:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=15, print_progress=False, print_crit_pairs_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))

Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 4, -4], [2, -4, 4], [8, 1, 1, 2, 2, 3, 3, 4, 4]]
Trying shortlex:
Max rounds reached: 15
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 15
Terminated by reaching max_rounds.
Total number of rules: 46
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 3, -3], [2, -3, 3], [2, 4, -4], [2, -4, 4], [4, 2, 3, 3, 4, -2, -1, -1, -4], [4, 1, 2, 2, 3, -1, -4, -4, -3], [4, -4, -4, -3, -3, 1, 1, 2, 2], [4, 2, 2, 3, 3, -1, -1, -4, -4], [4, 3, 3, 4, 4, -2, -2, -1, -1], [5, -2, -2, -1, -1, -4, 3, 3, 4], [7, 2, 2, 3, -2, -2, -1, -1, -1, -1, -4, -4, 3, 4, 4], [5, -1, -1, -4, -4, -3, 2, 2, 3], [5, -2, -1, -1, -4, -4, 2, 3, 3], [5, -3, -2, -2, -1, -1, 3, 4, 4], [7, 3, 4, 4, -1, -4, -4, -3, -3, -2, -2, -1, 2, 2, 3], [4, 3, 4, 4, 1, -3, -2, -2, -1], [4, 4, 1, 1, 2, -4, -3, -3, -2], [8, -2, -1, -1, -4, 1, 1, 2, 2, 2, 3, 3, -4, -3, -3], [11, -2, -1, -1, -4, 1, 1, 2, 

## Burnside groups

In [2]:
def generate_group_words(words,number_gens=2,max_length=4, exact_length=False, remove_cyc_reducible=True, remove_shifts=True, remove_inverses=True):
  ''' generates cyclically reduced group words on number_gens group generators up to length max_length '''
  words.clear()
  for i in range(1,number_gens+1):
    words.append([i])
    words.append([-i])
  # print("Added words of length 1.")
  for length in range(2,max_length+1):
    # print("Processing length", length)
    new_words = []
    for word in words:
      # print(word)
      if len(word) == length-1:
        for i in range(1,number_gens+1):
          if word[-1] != i:
            new_words.append(word + [-i])
          if word[-1] != -i:
            new_words.append(word + [i])
    # print(new_words)
    words+= new_words

  # print(words)
  if exact_length == True:
    for word in words:
      # print("looking at", word)
      if len(word) != max_length:
        # print("removing", word)
        words.remove(word)
    if remove_inverses == True:
      for word in words:
        inverse_word = group_inverse(word)
        if inverse_word in words:
          # print("Removing inverse of", word, ":", inverse_word)
          words.remove(inverse_word)
          # print(len(words), "words left.")
    if remove_shifts == True:
      for word in words:
        for i in range(1,len(word)):
          shift_word = word[i:] + word[:i]
          if shift_word != word and shift_word in words:
            # print("Removing shift of", word, ":", shift_word)
            words.remove(shift_word)
    return 0

  if remove_cyc_reducible == True:
    i=0
    while i < len(words):
      if len(words[i]) > 2 and words[i][0]==-words[i][-1]:
        words.remove(words[i])
      else:
        i+=1

  if remove_inverses == True:
    for word in words:
      inverse_word = group_inverse(word)
      if inverse_word in words:
        # print("Removing inverse of", word, ":", inverse_word)
        words.remove(inverse_word)
        # print(len(words), "words left.")

  if remove_shifts == True:
    for word in words:
      for i in range(1,len(word)):
        shift_word = word[i:] + word[:i]
        if shift_word != word and shift_word in words:
          # print("Removing shift of", word, ":", shift_word)
          words.remove(shift_word)

  if remove_shifts and remove_inverses:
    for word in words:
      inverse_word = group_inverse(word)
      for i in range(0,len(word)):
        shift_word = inverse_word[i:] + inverse_word[:i]
        if shift_word != word and shift_word in words:
          words.remove(shift_word)

  return 0

def generate_all_words(words,number_gens=2,max_length=4, exact_length=False):
  words.clear()
  for i in range(1,number_gens+1):
    words.append([i])
    words.append([-i])
  # print("Added words of length 1.")
  for length in range(2,max_length+1):
    # print("Processing length", length)
    new_words = []
    for word in words:
      # print(word)
      if len(word) == length-1:
        for i in range(1,number_gens+1):
          new_words.append(word + [-i])
          new_words.append(word + [i])
    # print(new_words)
    words+= new_words

def presentation_to_rules(relators):
  rules = []
  for relator in relators:
    rules.append( [len(relator)] + relator )
  return rules

In [ ]:
def detect_inverses(words):
  for word in words:
    if group_inverse(word) in words:
      return True
  return False

def detect_duplicates(words):
  for word in words:
    if words.count(word) > 1:
      return True
  return False

def detect_free_inclusion(word,words):
  word_found = False
  cancellation_made = True
  while len(word) > 1 and cancellation_made:
    cancellation_made = False
    for i in range(len(word)-1):
      # print(word)
      if word[i]==-word[i+1]:
        cancellation_made = True
        word = word[:i] + ( word[i+2:] if i+2<len(word) else [] )
        if word == []:
          return True
        break

  while len(word) > 1 and word[0] == -word[-1]:
    # print(word, "-->", word[1:-1])
    word = word[1:-1]
    # print(word)
    if word == []:
      return True

  for i in range(len(word)):
    if word[i:]+word[:i] in words:
      return True

  word = group_inverse(word)
  for i in range(len(word)):
    if word[i:]+word[:i] in words:
      return True
  return False


In [ ]:
group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying shortlex:")
rounds = knuth_bendix(rules, order=shortlex, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying shortlex with Chenadec order:")
rounds = knuth_bendix(rules, order=shortlex_with_chenadec, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying shortlex with positive first:")
rounds = knuth_bendix(rules, order=shortlex_with_positive_first, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying recursive path order:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying recursive path order with Chenadec order:")
rounds = knuth_bendix(rules, order=recursive_path_order_with_chenadec, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

group_words = []
generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*3 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
# print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,3). Starting rules:", rules)
print("Trying recursive path order with positive first order:")
rounds = knuth_bendix(rules, order=recursive_path_order_with_positive_first, reduce_crit_immediately=True, max_rounds=4, print_progress=False)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")

B(2,3). Starting rules: [[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [3, 1, 1, 1], [3, 2, 2, 2], [6, 1, 1, 1, 1, 1, 1], [6, 1, -2, 1, -2, 1, -2], [6, 1, 2, 1, 2, 1, 2], [6, 2, 2, 2, 2, 2, 2], [9, 1, 1, 1, 1, 1, 1, 1, 1, 1], [9, 1, 1, -2, 1, 1, -2, 1, 1, -2], [9, 1, 1, 2, 1, 1, 2, 1, 1, 2], [9, 1, -2, -2, 1, -2, -2, 1, -2, -2], [9, 1, 2, 2, 1, 2, 2, 1, 2, 2], [9, 2, 2, 2, 2, 2, 2, 2, 2, 2]]
Trying shortlex:
Max rounds reached: 4
Making shortcuts and removing redundancies, then stopping.
Stopping because max rounds reached: 4
Terminated by reaching max_rounds.
Total number of rules: 26
[[2, 1, -1], [2, -1, 1], [2, 2, -2], [2, -2, 2], [2, 2, 2, -2], [2, 1, 1, -1], [2, -1, -1, 1], [4, -2, 1, 2, -1, -1, -2, 1, 2], [4, -2, -1, 2, 1, -1, 2, 1, -2], [4, 1, 2, 1, -2, -2, -1, 2], [4, -2, -1, 2, -1, 2, 1, -2], [4, 2, 1, -2, 1, -2, -1, 2], [2, -2, -2, 2], [4, 1, 2, -1, -2, -1, -2, 1, 2], [4, 1, -2, -1, 2, -1, 2, 1, -2], [3, -2, 1, -2, -1, 2, -1], [3, 2, -1, 2, 1, -2, 1], [4, -2, 1, 2, 1, 2, -1

In [ ]:
group_words = []
generate_group_words(group_words,number_gens=2,max_length=4,exact_length=False,remove_cyc_reducible=True,\
                     remove_shifts=True, remove_inverses=True)
# generate_group_words(group_words,number_gens=2,max_length=3,exact_length=False,remove_cyc_reducible=False,\
#                      remove_shifts=False, remove_inverses=False)
all_starting_relators = [word*4 for word in group_words]
all_starting_rules = presentation_to_rules(all_starting_relators)
print(len(all_starting_rules))
rules = all_starting_rules

add_free_group_rules(rules)

print("B(2,4). Starting rules:", rules)
print("Trying recursive path order:")
rounds = knuth_bendix(rules, order=recursive_path_order, reduce_crit_immediately=True, max_rounds=4, print_progress=True)
if rounds != -1:
  print("Finished in", rounds, "rounds.\nTotal number of rules:", len(rules))
else:
  print("Terminated by reaching max_rounds.\nTotal number of rules:", len(rules))
print(rules)
print("Confluence:", check_confluence(rules,print_progress=False))
print("---------")


In [3]:
#include <iostream>
#include <vector>
#include <utility>
#include <list> // Assuming you still need list

// Helper function to print a vector of integers
void print_vector(const std::vector<int>& vec) {
    std::cout << "{";
    for (size_t i = 0; i < vec.size(); ++i) {
        std::cout << vec[i] << (i == vec.size() - 1 ? "" : ", ");
    }
    std::cout << "}";
}

// Helper function to print a pair of vectors of integers
void print_pair_of_vectors(const std::pair<std::vector<int>, std::vector<int>>& p) {
    std::cout << "{ ";
    print_vector(p.first);
    std::cout << ", ";
    print_vector(p.second);
    std::cout << " }";
}

int main() {
    std::list<std::pair<std::vector<int>, std::vector<int>>> rule_list;

    std::vector<int> word1 = {1, 2, -1};
    std::vector<int> word2 = {3, 4, 3};
    rule_list.push_back({word1, word2});

    if (!rule_list.empty()) {
        std::cout << "First element (pair of vectors): ";
        print_pair_of_vectors(rule_list.front());
        std::cout << std::endl;
    }

    std::cout << "Doubly linked list of word pairs initialized." << std::endl;

    return 0;
}

SyntaxError: invalid syntax (2380952503.py, line 6)